Team coursework: LightGBM classification on an anonymised Kaggle-style tabular dataset (class-imbalanced binary target), part of the same assignment as `svm_imbalance_and_tuning.ipynb` in this folder. The input CSV and rendered outputs are not included, and results are not independently reproduced in this portfolio copy.


### Biz Understanding
Obj: identify customers who are likely to apply for business credit in the future (target = 1)

### Data Understanding
- 100k records
- 200 anonymized numerical variables
- ID_code >> Customer Id
- Binary target <br>
  0=cust will not apply for business credit <br>
  1=cust will apply for business credit

### Data prep
- data type >> Numerical
- id col >> drop
- na value >> drop var_45 / others - use value imputation algo (means)
- outlliers >> will not delete
- correlation >> no correlation btw columns
- check dup record >> no
- features selection
- Check balance        <br>
target=0 >> 89962   <br>
target=1 >> 10038      <br>
~ 9:1  <br>

### Balancing tech
None, EasyEnsemble, SMOTE, Class-weight

### Modelling
LGBM

### Evaluation
PR-AUC, MCC, f2-score, confusion metrix, accuracy, precision, recall, AUC-ROC

FN - missed opportunity (opportunity cost)
FP - wasted marketing (marketing cost)
TP - our profit


-------------

step: EDA > drop column > partition (80/20) > imputation (mean) > data imbalancing & modelling



# Import Lib

In [ ]:
# First cell in notebook
import warnings
warnings.filterwarnings('ignore')

import sys
import os

# Redirect stderr temporarily
import io
sys.stderr = io.StringIO()

# Or completely disable warnings
def noop(*args, **kwargs):
    pass

warnings.warn = noop
warnings.warn_explicit = noop
warnings.showwarning = noop

In [ ]:
!pip install lightgbm

In [ ]:
!pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib as plt
import matplotlib.pyplot as plt

from sklearn.metrics import precision_recall_curve, fbeta_score, average_precision_score
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import EasyEnsembleClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import RandomOverSampler

import optuna
import warnings
import time

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Or more specifically
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Import data

In [ ]:
# Import data
data = pd.read_csv("data/datafile_small.csv")

data.head()

# Verify data

In [ ]:
# check var structure
data.info()

# check stat summary
data.describe(include='all')

In [ ]:
# check na
null_counts = data.isnull().sum()
columns_with_nulls = null_counts[null_counts > 0]
print("null_counts:")
print(null_counts)
print("\ncolumns_with_nulls:")
print(columns_with_nulls)

columns_with_nulls_50up = null_counts[null_counts > 50] ## replace with mean??
print("\ncolumns_with_nulls > 50:")
print(columns_with_nulls_50up)

In [ ]:
# check correlations >> can drop var_45
correlations = data.corr()['target']
print(correlations)
correlations.mean()

high_corr = correlations > 0.05
if high_corr.any():  # Check if any values are True
    print(correlations[high_corr])

print("\ncorrelations of var_45:")
print(correlations.var_45)
print("\ncorrelations of var_76:")
print(correlations.var_76)
print("\ncorrelations of var_96:")
print(correlations.var_96)

# Clean data

In [ ]:
# drop column ID_code and var_45 only in train set
data.drop(columns=['ID_code'], inplace=True)
data.drop(columns=['var_45'], inplace=True)

data.head()

# Partition and Imputation

In [ ]:
# Partition
random_seed = 30
X = data.drop(columns=['target'])
y = data['target']

# train80, test20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=random_seed)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
# Single imputation
imputer = SimpleImputer(strategy='mean')

# Fitting
imputer.fit(X_train)

# Imputed train data set (convert back to DataFrame to preserve column names)
X_train_imputed = pd.DataFrame(
    imputer.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Balancing and Modelling

## check imbalance

In [ ]:
# check data imbalance
print(f"\ndiagnosis distribution in training set:\n{y_train.value_counts(normalize=True)}")
print(f"\ndiagnosis distribution in test set:\n{y_test.value_counts(normalize=True)}")

## Baseline model (no parameter tuning)

In [ ]:
# Define F2-Score function
def f2_score(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

### Balancing: None

In [ ]:
# METHOD 1: BASELINE (NO BALANCING)
print("\n" + "="*80)
print("METHOD 1: BASELINE (NO BALANCING)")
print("="*80)

start_time = time.time()

# Create model
model_no_balance = lgb.LGBMClassifier(
    random_state=random_seed,
    n_estimators=100,
    learning_rate=0.1,
    verbose=-1
)

# Train model
model_no_balance.fit(X_train_imputed, y_train)

# Make predictions
y_pred_no_balance = model_no_balance.predict(X_test_imputed)
y_pred_proba_no_balance = model_no_balance.predict_proba(X_test_imputed)[:, 1]

training_time = time.time() - start_time

# Calculate PR-AUC
# precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba_no_balance)
# pr_auc_baseline = auc(recall_vals, precision_vals)
pr_auc_baseline = average_precision_score(y_test, y_pred_proba_no_balance)

# Store results
results_noBalance = {
    'Method': 'Baseline (No Balancing)',
    'Train_Size': len(y_train),
    'Training_Time': training_time,
    'Accuracy': accuracy_score(y_test, y_pred_no_balance),
    'Precision': precision_score(y_test, y_pred_no_balance),
    'Recall': recall_score(y_test, y_pred_no_balance),
    'F2_Score': f2_score(y_test, y_pred_no_balance),
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_no_balance),
    'MCC': matthews_corrcoef(y_test, y_pred_no_balance),
    'PR_AUC': pr_auc_baseline
}

print(f"\n✅ Baseline trained in {results_noBalance['Training_Time']:.2f}s")
print(f"\nResults:")
print(f"   Accuracy:  {results_noBalance['Accuracy']:.4f}")
print(f"   Precision: {results_noBalance['Precision']:.4f}")
print(f"   Recall:    {results_noBalance['Recall']:.4f}")
print(f"   F2-Score:  {results_noBalance['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_noBalance['ROC_AUC']:.4f}")
print(f"   MCC:       {results_noBalance['MCC']:.4f}")
print(f"   PR_AUC:    {results_noBalance['PR_AUC']:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_no_balance)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_no_balance))

### Balancing: EasyEnsemble

In [ ]:
from sklearn.metrics import fbeta_score, auc as calculate_auc

# METHOD 2: EASYENSEMBLE
print("\n" + "="*80)
print("METHOD 2: EASYENSEMBLE (ALL FEATURES)")
print("="*80)

start_time = time.time()

easy_ensemble = EasyEnsembleClassifier(
    estimator=lgb.LGBMClassifier(
        random_state=random_seed,
        n_estimators=100,
        learning_rate=0.1,
        verbose=-1
    ),
    n_estimators=10,
    random_state=random_seed,
    n_jobs=-1
)

# Train
easy_ensemble.fit(X_train_imputed, y_train)

# Predict
y_pred_ee = easy_ensemble.predict(X_test_imputed)
y_pred_proba_ee = easy_ensemble.predict_proba(X_test_imputed)[:, 1]

training_time = time.time() - start_time

# Calculate PR-AUC
pr_auc_EE = average_precision_score(y_test, y_pred_proba_ee)

# Store results
results_EE = {
    'Method': f'EasyEnsemble (All Features)',
    'Train_Size': len(y_train),
    'N_Features': len(X_train_imputed.columns),
    'Training_Time': training_time,
    'Accuracy': accuracy_score(y_test, y_pred_ee),
    'Precision': precision_score(y_test, y_pred_ee),
    'Recall': recall_score(y_test, y_pred_ee),
    'F2_Score': fbeta_score(y_test, y_pred_ee, beta=2),  # Use fbeta_score directly
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_ee),
    'PR_AUC': pr_auc_EE,
    'MCC': matthews_corrcoef(y_test, y_pred_ee)
}

print(f"\n✅ EasyEnsemble trained in {results_EE['Training_Time']:.2f}s")
print(f"\nResults:")
print(f"   Accuracy:  {results_EE['Accuracy']:.4f}")
print(f"   Precision: {results_EE['Precision']:.4f}")
print(f"   Recall:    {results_EE['Recall']:.4f}")
print(f"   F2-Score:  {results_EE['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_EE['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_EE['PR_AUC']:.4f}")
print(f"   MCC:       {results_EE['MCC']:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_ee)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ee))

### Balancing: Class-weight

In [ ]:
# METHOD 3: Class-weight
print("\n" + "="*80)
print("METHOD 3: CLASS WEIGHT (BALANCED)")
print("="*80)

start_time = time.time()

model_cw = lgb.LGBMClassifier(
    random_state=random_seed,
    class_weight='balanced',  # Automatically balance classes
    n_estimators=100,
    learning_rate=0.1,
    verbose=-1
)

model_cw.fit(X_train_imputed, y_train)
y_pred_cw = model_cw.predict(X_test_imputed)
y_pred_proba_cw = model_cw.predict_proba(X_test_imputed)[:, 1]

training_time = time.time() - start_time

# Calculate PR-AUC
pr_auc_cw = average_precision_score(y_test, y_pred_proba_cw)

# Store results
results_cw = {
    'Method': 'Class Weight',
    'Train_Size': len(y_train),
    'Training_Time': training_time,
    'Accuracy': accuracy_score(y_test, y_pred_cw),
    'Precision': precision_score(y_test, y_pred_cw),
    'Recall': recall_score(y_test, y_pred_cw),
    'F2_Score': f2_score(y_test, y_pred_cw),
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_cw),
    'MCC': matthews_corrcoef(y_test, y_pred_cw),
    'PR_AUC': pr_auc_cw
}

print(f"\n✅ Class Weight trained in {results_cw['Training_Time']:.2f}s")
print(f"\nResults:")
print(f"   Accuracy:  {results_cw['Accuracy']:.4f}")
print(f"   Precision: {results_cw['Precision']:.4f}")
print(f"   Recall:    {results_cw['Recall']:.4f}")
print(f"   F2-Score:  {results_cw['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_cw['ROC_AUC']:.4f}")
print(f"   MCC:       {results_cw['MCC']:.4f}")
print(f"   PR_AUC:    {results_cw['PR_AUC']:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_cw)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cw))

### Balancing: SMOTE

In [ ]:
# METHOD 4: SMOTE
print("\n" + "="*80)
print("METHOD 4: SMOTE")
print("="*80)

start_time = time.time()

smote = SMOTE(random_state=random_seed)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imputed, y_train)

model_smote = lgb.LGBMClassifier(
    random_state=random_seed,
    n_estimators=100,
    learning_rate=0.1,
    verbose=-1
)

model_smote.fit(X_train_smote, y_train_smote)

# Predict
y_pred_smote = model_smote.predict(X_test_imputed)
y_pred_proba_smote = model_smote.predict_proba(X_test_imputed)[:, 1]

training_time = time.time() - start_time

# Calculate PR-AUC
pr_auc_smote = average_precision_score(y_test, y_pred_proba_smote)

# Store results
results_smote = {
    'Method': 'SMOTE',
    'Train_Size': len(y_train_smote),
    'Training_Time': training_time,
    'Accuracy': accuracy_score(y_test, y_pred_smote),
    'Precision': precision_score(y_test, y_pred_smote),
    'Recall': recall_score(y_test, y_pred_smote),
    'F2_Score': f2_score(y_test, y_pred_smote),
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_smote),
    'MCC': matthews_corrcoef(y_test, y_pred_smote),
    'PR_AUC': pr_auc_smote

}

print(f"\n✅ SMOTE trained in {results_smote['Training_Time']:.2f}s")
print(f"\nResults:")
print(f"   Accuracy:  {results_smote['Accuracy']:.4f}")
print(f"   Precision: {results_smote['Precision']:.4f}")
print(f"   Recall:    {results_smote['Recall']:.4f}")
print(f"   F2-Score:  {results_smote['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_smote['ROC_AUC']:.4f}")
print(f"   MCC:       {results_smote['MCC']:.4f}")
print(f"   PR_AUC:    {results_smote['PR_AUC']:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_smote)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

### Compare result of baseline model

In [ ]:
# Create results dictionary
results = {
    'Baseline': results_noBalance,
    'EasyEnsemble': results_EE,
    'Class Weight': results_cw,
    'SMOTE': results_smote
}

# ============================================================
# CREATE COMPARISON DATAFRAME
# ============================================================
print("\n" + "="*80)
print("RESULTS COMPARISON TABLE")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['Accuracy'] for m in results.keys()],
    'Precision': [results[m]['Precision'] for m in results.keys()],
    'Recall': [results[m]['Recall'] for m in results.keys()],
    'F2_Score': [results[m]['F2_Score'] for m in results.keys()],
    'ROC_AUC': [results[m]['ROC_AUC'] for m in results.keys()],
    'PR_AUC': [results[m]['PR_AUC'] for m in results.keys()],
    'MCC': [results[m]['MCC'] for m in results.keys()]
})

# Sort by PR-AUC (primary), then F2-Score (secondary), then MCC (tertiary)
comparison_df = comparison_df.sort_values(
    by=['PR_AUC', 'F2_Score', 'MCC'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Add rank column
comparison_df.insert(0, 'Rank', range(1, len(comparison_df) + 1))

print("\n" + comparison_df.to_string(index=False))

# ============================================================
# BEST MODEL ANALYSIS
# ============================================================
print("\n" + "="*80)
print("BEST MODEL ANALYSIS")
print("="*80)

# Get best model name from rank 1
best_model_name = comparison_df.iloc[0]['Model']
best_result = results[best_model_name]

print(f"\n🏆 BEST MODEL (Ranked by: PR-AUC > F2-Score > MCC): {best_model_name}")

print(f"\n   Primary Ranking Metrics:")
print(f"   ├─ PR-AUC:    {best_result['PR_AUC']:.4f} (1st priority)")
print(f"   ├─ F2-Score:  {best_result['F2_Score']:.4f} (2nd priority)")
print(f"   └─ MCC:       {best_result['MCC']:.4f} (3rd priority)")

print(f"\n   Complete Performance:")
print(f"   ├─ Accuracy:  {best_result['Accuracy']:.4f}")
print(f"   ├─ Precision: {best_result['Precision']:.4f}")
print(f"   ├─ Recall:    {best_result['Recall']:.4f}")
print(f"   └─ ROC-AUC:   {best_result['ROC_AUC']:.4f}")

# Get confusion matrix from the appropriate predictions
print("\n   Confusion Matrix:")

if best_model_name == 'Baseline':
    cm = confusion_matrix(y_test, y_pred_no_balance)
elif best_model_name == 'EasyEnsemble':
    cm = confusion_matrix(y_test, y_pred_ee)
elif best_model_name == 'Class Weight':
    cm = confusion_matrix(y_test, y_pred_cw)
else:  # SMOTE
    cm = confusion_matrix(y_test, y_pred_smote)

print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\n   Breakdown:")
print(f"   ├─ True Negatives:  {tn:,}")
print(f"   ├─ False Positives: {fp:,}")
print(f"   ├─ False Negatives: {fn:,}")
print(f"   └─ True Positives:  {tp:,}")

# Calculate additional metrics
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print(f"\n   Additional Metrics:")
print(f"   ├─ Specificity (TNR): {specificity:.4f}")
print(f"   ├─ Sensitivity (TPR): {sensitivity:.4f}")
print(f"   ├─ False Positive Rate: {fpr:.4f}")
print(f"   └─ False Negative Rate: {fnr:.4f}")

# ============================================================
# HIGHLIGHT BEST SCORES FOR EACH METRIC
# ============================================================
print("\n" + "="*80)
print("BEST SCORES BY METRIC")
print("="*80)

metrics = ['Accuracy', 'Precision', 'Recall', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']
for metric in metrics:
    best_idx = comparison_df[metric].idxmax()
    best_model = comparison_df.loc[best_idx, 'Model']
    best_score = comparison_df.loc[best_idx, metric]
    print(f"{metric:<15}: {best_model:<25} ({best_score:.4f})")

# ============================================================
# ALTERNATIVE RANKINGS
# ============================================================
print("\n" + "="*80)
print("ALTERNATIVE RANKINGS")
print("="*80)

print("\n1. Current ranking (PR-AUC > F2 > MCC):")
for idx, row in comparison_df.iterrows():
    marker = "🏆" if row['Rank'] == 1 else f"  {row['Rank']}."
    print(f"   {marker} {row['Model']:<20} (PR-AUC: {row['PR_AUC']:.4f})")

print("\n2. If ranked by ROC-AUC:")
roc_sorted = comparison_df.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
for idx, row in roc_sorted.iterrows():
    marker = "🏆" if idx == 0 else f"  {idx+1}."
    print(f"   {marker} {row['Model']:<20} (ROC-AUC: {row['ROC_AUC']:.4f})")

print("\n3. If ranked by F2-Score:")
f2_sorted = comparison_df.sort_values('F2_Score', ascending=False).reset_index(drop=True)
for idx, row in f2_sorted.iterrows():
    marker = "🏆" if idx == 0 else f"  {idx+1}."
    print(f"   {marker} {row['Model']:<20} (F2: {row['F2_Score']:.4f})")

print("\n4. If ranked by Recall:")
recall_sorted = comparison_df.sort_values('Recall', ascending=False).reset_index(drop=True)
for idx, row in recall_sorted.iterrows():
    marker = "🏆" if idx == 0 else f"  {idx+1}."
    print(f"   {marker} {row['Model']:<20} (Recall: {row['Recall']:.4f})")

# Check if different rankings produce different winners
current_winner = comparison_df.iloc[0]['Model']
roc_winner = roc_sorted.iloc[0]['Model']
f2_winner = f2_sorted.iloc[0]['Model']

if current_winner != roc_winner or current_winner != f2_winner:
    print(f"\n⚠️  Note: Different ranking methods produce different winners!")
    print(f"   Current (PR-AUC): {current_winner}")
    print(f"   ROC-AUC ranking:  {roc_winner}")
    print(f"   F2 ranking:       {f2_winner}")
    print(f"\n   💡 For imbalanced data, PR-AUC is generally more reliable than ROC-AUC.")
else:
    print(f"\n✓ All ranking methods agree: {current_winner} is the best model.")

# ============================================================
# DETAILED COMPARISON - ALL MODELS
# ============================================================
print("\n" + "="*80)
print("DETAILED COMPARISON - ALL MODELS")
print("="*80)

for rank, (idx, row) in enumerate(comparison_df.iterrows(), 1):
    model_name = row['Model']
    print(f"\n{rank}. {model_name}")
    print("-" * 60)
    print(f"   Accuracy:  {row['Accuracy']:.4f}")
    print(f"   Precision: {row['Precision']:.4f}")
    print(f"   Recall:    {row['Recall']:.4f}")
    print(f"   F2-Score:  {row['F2_Score']:.4f}")
    print(f"   ROC-AUC:   {row['ROC_AUC']:.4f}")
    print(f"   PR-AUC:    {row['PR_AUC']:.4f} ★")
    print(f"   MCC:       {row['MCC']:.4f}")

In [ ]:
print("="*80)
print("PRECISION-RECALL CURVE VISUALIZATION")
print("="*80)

# ============================================================
# SINGLE PLOT - ALL METHODS
# ============================================================

plt.figure(figsize=(12, 8))

# Store predictions for all methods
predictions_dict = {
    'Baseline': y_pred_proba_no_balance,
    'EasyEnsemble': y_pred_proba_ee,
    'Class Weight': y_pred_proba_cw,
    'SMOTE': y_pred_proba_smote
}

colors = ['blue', 'red', 'green', 'orange']
line_styles = ['-', '--', '-.', ':']

# Calculate baseline (random classifier)
baseline_precision = y_test.sum() / len(y_test)

# Plot baseline
plt.axhline(y=baseline_precision, color='gray', linestyle='--',
           linewidth=2, label=f'Random Classifier (AP={baseline_precision:.4f})', alpha=0.5)

# Plot PR curves for all methods
for (method, y_proba), color, linestyle in zip(predictions_dict.items(), colors, line_styles):
    # Calculate precision-recall curve
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

    # Calculate PR-AUC
    pr_auc = auc(recall, precision)

    # Calculate Average Precision (AP)
    ap = average_precision_score(y_test, y_proba)

    # Plot
    plt.plot(recall, precision, color=color, linewidth=2.5, linestyle=linestyle,
            label=f'{method} (PR-AUC={pr_auc:.4f}, AP={ap:.4f})', alpha=0.8)

    # Mark the operating point (default threshold = 0.5)
    y_pred_temp = (y_proba >= 0.5).astype(int)
    precision_at_05 = precision_score(y_test, y_pred_temp)
    recall_at_05 = recall_score(y_test, y_pred_temp)
    plt.scatter([recall_at_05], [precision_at_05], s=150, color=color,
               edgecolor='black', linewidth=2, zorder=5, marker='o')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall (Sensitivity)', fontsize=13)
plt.ylabel('Precision (PPV)', fontsize=13)
plt.title('Precision-Recall Curves - All Methods\n(● = Operating Point at threshold=0.5)',
         fontsize=15)
plt.legend(loc='best', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()

## Feature selection - EasyEnsemble

### find feature importance and best threshold

In [ ]:
from sklearn.metrics import precision_recall_curve, auc as calculate_auc

print("="*80)
print("TESTING MULTIPLE GAIN SCORE THRESHOLDS")
print("="*80)

# Extract feature importances
feature_importances_list = []
for pipeline in easy_ensemble.estimators_:
    classifier = pipeline.steps[-1][1]
    feature_importances_list.append(classifier.feature_importances_)

avg_importance = np.mean(feature_importances_list, axis=0)

feature_importance = pd.DataFrame({
    'feature': X_train_imputed.columns,
    'gain_score': avg_importance
}).sort_values('gain_score', ascending=False)

print(f"\nTotal features available: {len(feature_importance)}")
print(f"Feature importance range: {feature_importance['gain_score'].min():.2f} to {feature_importance['gain_score'].max():.2f}")

# Test different thresholds
thresholds = [100, 70, 50, 30, 20, 10, 5, 1]

results = []

for threshold in thresholds:
    selected_features = feature_importance[feature_importance['gain_score'] > threshold]['feature'].tolist()

    if len(selected_features) == 0:
        print(f"\nThreshold > {threshold}: No features selected, skipping...")
        continue

    print(f"\nTesting threshold > {threshold}: {len(selected_features)} features")

    # Filter data
    X_train_temp = X_train_imputed[selected_features]
    X_test_temp = X_test_imputed[selected_features]

    # Train model
    start_time = time.time()
    model_temp = EasyEnsembleClassifier(
        estimator=lgb.LGBMClassifier(
            random_state=random_seed,
            n_estimators=100,
            learning_rate=0.1,
            verbose=-1
        ),
        n_estimators=10,
        random_state=random_seed,
        n_jobs=-1
    )

    model_temp.fit(X_train_temp, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred_temp = model_temp.predict(X_test_temp)
    y_pred_proba_temp = model_temp.predict_proba(X_test_temp)[:, 1]

    # Calculate metrics
    roc_auc = roc_auc_score(y_test, y_pred_proba_temp)

    # **FIXED - Calculate PR-AUC for each threshold using its own predictions**
    pr_auc = average_precision_score(y_test, y_pred_proba_temp)

    # Additional metrics
    accuracy = accuracy_score(y_test, y_pred_temp)
    precision = precision_score(y_test, y_pred_temp)
    recall = recall_score(y_test, y_pred_temp)
    f2 = fbeta_score(y_test, y_pred_temp, beta=2)

    results.append({
        'threshold': threshold,
        'n_features': len(selected_features),
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f2_score': f2,
        'training_time': training_time,
        'feature_reduction_%': (1 - len(selected_features) / len(feature_importance)) * 100
    })

    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print(f"  PR-AUC:    {pr_auc:.4f}")
    print(f"  F2-Score:  {f2:.4f}")
    print(f"  Time:      {training_time:.2f}s")

# Create results dataframes
results_df_ROCAUC = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
results_df_PRAUC = pd.DataFrame(results).sort_values('pr_auc', ascending=False)

# ============================================================
# RESULTS RANKED BY PR-AUC
# ============================================================
print("\n" + "="*80)
print("RESULTS RANKED BY PR-AUC (RECOMMENDED FOR IMBALANCED DATA)")
print("="*80)
print("\n" + results_df_PRAUC.to_string(index=False))

best_result_PRAUC = results_df_PRAUC.iloc[0]

print("\n" + "="*60)
print("🏆 BEST THRESHOLD (by PR-AUC)")
print("="*60)
print(f"  Threshold:           > {best_result_PRAUC['threshold']}")
print(f"  Number of features:  {int(best_result_PRAUC['n_features'])} / {len(feature_importance)}")
print(f"  PR-AUC:              {best_result_PRAUC['pr_auc']:.4f}")
print(f"  ROC-AUC:             {best_result_PRAUC['roc_auc']:.4f}")
print(f"  F2-Score:            {best_result_PRAUC['f2_score']:.4f}")
print(f"  Feature reduction:   {best_result_PRAUC['feature_reduction_%']:.2f}%")

### train model with selected features (based on PR-AUC)

In [ ]:
print("\n" + "="*80)
print("MODEL: FEATURE SELECTION BY PR-AUC THRESHOLD")
print("="*80)

GAIN_THRESHOLD_PRAUC = best_result_PRAUC['threshold']

# Select features
selected_features_prauc = feature_importance[feature_importance['gain_score'] > GAIN_THRESHOLD_PRAUC]['feature'].tolist()

print(f"\nTotal features: {len(feature_importance)}")
print(f"Gain threshold: {GAIN_THRESHOLD_PRAUC}")
print(f"Selected features: {len(selected_features_prauc)}")
print(f"Reduction: {(1 - len(selected_features_prauc)/len(feature_importance))*100:.1f}%")

# Filter datasets
X_train_selected_prauc = X_train_imputed[selected_features_prauc]
X_test_selected_prauc = X_test_imputed[selected_features_prauc]

start_time = time.time()

# Train model
model_selected_prauc = EasyEnsembleClassifier(
    estimator=lgb.LGBMClassifier(
        random_state=random_seed,
        n_estimators=100,
        learning_rate=0.1,
        verbose=-1
    ),
    n_estimators=10,
    random_state=random_seed,
    n_jobs=-1
)

model_selected_prauc.fit(X_train_selected_prauc, y_train)
training_time_prauc = time.time() - start_time

# Predict
y_pred_selected_prauc = model_selected_prauc.predict(X_test_selected_prauc)
y_pred_proba_selected_prauc = model_selected_prauc.predict_proba(X_test_selected_prauc)[:, 1]

# Calculate PR-AUC using average_precision_score
pr_auc_selected_prauc = average_precision_score(y_test, y_pred_proba_selected_prauc)

# Store results
results_selected_prauc = {
    'Method': f'PR-AUC Selected features',
    'Train_Size': len(y_train),
    'N_Features': len(selected_features_prauc),
    'Training_Time': training_time_prauc,
    'Accuracy': accuracy_score(y_test, y_pred_selected_prauc),
    'Precision': precision_score(y_test, y_pred_selected_prauc),
    'Recall': recall_score(y_test, y_pred_selected_prauc),
    'F2_Score': fbeta_score(y_test, y_pred_selected_prauc, beta=2),
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_selected_prauc),
    'PR_AUC': pr_auc_selected_prauc,
    'MCC': matthews_corrcoef(y_test, y_pred_selected_prauc)
}

print(f"\n✅ Model trained in {training_time_prauc:.2f}s")
print(f"\nResults:")
print(f"   Features:  {len(selected_features_prauc)}")
print(f"   Accuracy:  {results_selected_prauc['Accuracy']:.4f}")
print(f"   Precision: {results_selected_prauc['Precision']:.4f}")
print(f"   Recall:    {results_selected_prauc['Recall']:.4f}")
print(f"   F2-Score:  {results_selected_prauc['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_selected_prauc['ROC_AUC']:.4f}")
print(f"   MCC:       {results_selected_prauc['MCC']:.4f}")
print(f"   PR-AUC:    {results_selected_prauc['PR_AUC']:.4f}")

print("\nConfusion Matrix:")
cm_prauc = confusion_matrix(y_test, y_pred_selected_prauc)
print(cm_prauc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_selected_prauc))

### compare result
easyEssemble <br>
easyEssemble + FS



summary: EasyEnsemble with feature selection gives better PR_AUC value

In [ ]:
# ============================================================
# COMPREHENSIVE COMPARISON
# ============================================================
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON - FEATURE SELECTION APPROACHES")
print("="*80)

# Combine all results
all_results_fs = [ results_selected_prauc, results_EE]

# Create comparison dataframe
comparison_fs_df = pd.DataFrame(all_results_fs)

# Sort by PR-AUC
comparison_fs_df = comparison_fs_df.sort_values('PR_AUC', ascending=False).reset_index(drop=True)
comparison_fs_df.insert(0, 'Rank', range(1, len(comparison_fs_df) + 1))

print("\n" + comparison_fs_df.to_string(index=False))



In [ ]:
import joblib

# selected_features 저장
joblib.dump(selected_features_prauc, 'lgbm_selected_features.pkl')
print(f"✓ Saved: lgbm_selected_features.pkl ({len(selected_features_prauc)} features)")

## Hyperarameter tuning - EasyEnsemble + FS

### Optuna - EasyEnsemble + feature selected (based on PR-AUC)

In [ ]:
from sklearn.metrics import make_scorer, average_precision_score
from sklearn.model_selection import cross_val_score
import optuna
import time

print("="*80)
print("OPTUNA OPTIMIZATION WITH FEATURE SELECTION")
print("="*80)

# ============================================================
# FEATURE SELECTION
# ============================================================
print("\n" + "="*60)
print("APPLYING FEATURE SELECTION")
print("="*60)

# Use the best threshold found earlier (from PR-AUC ranking)
GAIN_THRESHOLD = best_result_PRAUC['threshold']

print(f"Using threshold: > {GAIN_THRESHOLD}")
print(f"Original features: {len(X_train_imputed.columns)}")

# Select features based on threshold
selected_features = feature_importance[feature_importance['gain_score'] > GAIN_THRESHOLD]['feature'].tolist()

print(f"Selected features: {len(selected_features)}")
print(f"Feature reduction: {(1 - len(selected_features)/len(X_train_imputed.columns))*100:.1f}%")

# Filter datasets with selected features
X_train_selected = X_train_imputed[selected_features]
X_test_selected = X_test_imputed[selected_features]

# ============================================================
# OPTUNA OBJECTIVE FUNCTION
# ============================================================
def objective_cv_selected(trial):
    params = {
        # EasyEnsemble parameters
        'ee_n_estimators': trial.suggest_int('ee_n_estimators', 5, 30),
        'ee_sampling_strategy': trial.suggest_categorical('ee_sampling_strategy', ['auto', 0.5, 0.7, 0.8, 1.0]),

        # LightGBM parameters
        'lgb_n_estimators': trial.suggest_int('lgb_n_estimators', 50, 500),
        'lgb_learning_rate': trial.suggest_float('lgb_learning_rate', 0.01, 0.2, log=True),
        'lgb_max_depth': trial.suggest_int('lgb_max_depth', 3, 10),
        'lgb_num_leaves': trial.suggest_int('lgb_num_leaves', 10, 100)
    }

    # Create model
    model = EasyEnsembleClassifier(
        estimator=lgb.LGBMClassifier(
            random_state=random_seed,
            n_estimators=params['lgb_n_estimators'],
            learning_rate=params['lgb_learning_rate'],
            max_depth=params['lgb_max_depth'],
            num_leaves=params['lgb_num_leaves'],
            verbose=-1
        ),
        n_estimators=params['ee_n_estimators'],
        sampling_strategy=params['ee_sampling_strategy'],
        random_state=random_seed,
        n_jobs=-1
    )

    # Cross-validation with PR-AUC on SELECTED FEATURES
    cv_scores = cross_val_score(
        model,
        X_train_selected,  # Use selected features
        y_train,
        cv=3,
        scoring='average_precision',  # PR-AUC
        n_jobs=-1
    )

    return cv_scores.mean()

# ============================================================
# RUN OPTUNA OPTIMIZATION
# ============================================================
n_trials = 30

print("\n" + "="*60)
print("STARTING OPTUNA OPTIMIZATION")
print("="*60)
print(f"Trials:           {n_trials}")
print(f"CV Folds:         3")
print(f"Scoring:          PR-AUC (average_precision)")
print(f"Features:         {len(selected_features)} (selected)")

start_time = time.time()

# Create study
study_selected = optuna.create_study(
    direction='maximize',
    study_name='easyensemble_feature_selected',
    sampler=optuna.samplers.TPESampler(seed=random_seed)
)

study_selected.optimize(objective_cv_selected, n_trials=n_trials, show_progress_bar=True)

optuna_time = time.time() - start_time

print(f"\n✅ Optuna completed in {optuna_time:.2f}s ({optuna_time/60:.1f} minutes)")

# ============================================================
# OPTIMIZATION RESULTS
# ============================================================
print("\n" + "="*80)
print("OPTIMIZATION RESULTS (CV with Feature Selection)")
print("="*80)

print(f"\nNumber of trials:     {len(study_selected.trials)}")
print(f"Best CV PR-AUC:       {study_selected.best_value:.4f}")
print(f"Features used:        {len(selected_features)} / {len(X_train_imputed.columns)}")

# ============================================================
# BEST HYPERPARAMETERS
# ============================================================
print("\n" + "="*80)
print("BEST HYPERPARAMETERS")
print("="*80)

best_params_selected = study_selected.best_params

print("\nEasyEnsemble Parameters:")
ee_params = {k: v for k, v in best_params_selected.items() if k.startswith('ee_')}
for key, value in ee_params.items():
    print(f"  {key:<30}: {value}")

print("\nLightGBM Parameters:")
lgb_params = {k: v for k, v in best_params_selected.items() if k.startswith('lgb_')}
for key, value in lgb_params.items():
    print(f"  {key:<30}: {value}")



In [ ]:
# ============================================================
# TRAIN FINAL MODEL WITH BEST PARAMETERS
# ============================================================
print("\n" + "="*80)
print("TRAINING FINAL MODEL")
print("="*80)

start_time = time.time()

model_optuna_selected = EasyEnsembleClassifier(
    estimator=lgb.LGBMClassifier(
        random_state=random_seed,
        n_estimators=best_params_selected['lgb_n_estimators'],
        learning_rate=best_params_selected['lgb_learning_rate'],
        max_depth=best_params_selected['lgb_max_depth'],
        num_leaves=best_params_selected['lgb_num_leaves'],
        verbose=-1
    ),
    n_estimators=best_params_selected['ee_n_estimators'],
    sampling_strategy=best_params_selected['ee_sampling_strategy'],
    random_state=random_seed,
    n_jobs=-1
)

model_optuna_selected.fit(X_train_selected, y_train)
training_time = time.time() - start_time

# Predictions
y_pred_optuna_selected = model_optuna_selected.predict(X_test_selected)
y_pred_proba_optuna_selected = model_optuna_selected.predict_proba(X_test_selected)[:, 1]

# Calculate metrics
from sklearn.metrics import fbeta_score, auc as calculate_auc

# precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba_optuna_selected)
# pr_auc_optuna_selected = calculate_auc(recall_vals, precision_vals)
pr_auc_optuna_selected = average_precision_score(y_test, y_pred_proba_optuna_selected)

# Store results
results_optuna_selected = {
    'Method': 'Optuna + Feature Selection',
    'Train_Size': len(y_train),
    'N_Features': len(selected_features),
    'Training_Time': training_time,
    'Accuracy': accuracy_score(y_test, y_pred_optuna_selected),
    'Precision': precision_score(y_test, y_pred_optuna_selected),
    'Recall': recall_score(y_test, y_pred_optuna_selected),
    'F2_Score': fbeta_score(y_test, y_pred_optuna_selected, beta=2),
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_optuna_selected),
    'PR_AUC': pr_auc_optuna_selected,
    'MCC': matthews_corrcoef(y_test, y_pred_optuna_selected)
}

# ============================================================
# DISPLAY RESULTS
# ============================================================
print(f"\n✅ Model trained in {results_optuna_selected['Training_Time']:.2f}s")

print(f"\nTest Set Results:")
print(f"   Features:  {results_optuna_selected['N_Features']}")
print(f"   Accuracy:  {results_optuna_selected['Accuracy']:.4f}")
print(f"   Precision: {results_optuna_selected['Precision']:.4f}")
print(f"   Recall:    {results_optuna_selected['Recall']:.4f}")
print(f"   F2-Score:  {results_optuna_selected['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_optuna_selected['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_optuna_selected['PR_AUC']:.4f}")
print(f"   MCC:       {results_optuna_selected['MCC']:.4f}")

print("\nConfusion Matrix:")
cm_optuna_selected = confusion_matrix(y_test, y_pred_optuna_selected)
print(cm_optuna_selected)

tn, fp, fn, tp = cm_optuna_selected.ravel()
print(f"\n  TN: {tn:,} | FP: {fp:,}")
print(f"  FN: {fn:,} | TP: {tp:,}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_optuna_selected))


print("\n" + "="*80)
print("OPTUNA OPTIMIZATION WITH FEATURE SELECTION COMPLETE")
print("="*80)

print("\n📊 Summary:")
print(f"  • Optimization time:  {optuna_time/60:.1f} minutes")
print(f"  • Best CV PR-AUC:     {study_selected.best_value:.4f}")
print(f"  • Test PR-AUC:        {results_optuna_selected['PR_AUC']:.4f}")
print(f"  • Features used:      {len(selected_features)} / {len(X_train_imputed.columns)}")
print(f"  • Feature reduction:  {(1 - len(selected_features)/len(X_train_imputed.columns))*100:.1f}%")

In [ ]:
from joblib import dump

# Optuna 모델 저장
dump(model_optuna_selected, 'lgbm_optuna_best.pkl')
print("✓ Model saved: lgbm_optuna_best.pkl")

# Selected features 저장 (Optuna에서 사용한 것)
dump(selected_features, 'lgbm_selected_features.pkl')
print(f"✓ Features saved: {len(selected_features)} features")

# 예측 확률도 저장
y_proba_lgbm = model_optuna_selected.predict_proba(X_test_selected)[:, 1]
dump(y_proba_lgbm, 'lgbm_y_proba.pkl')
print("✓ Predictions saved: lgbm_y_proba.pkl")

### Grid Search - EasyEnsemble + feature selected (based on PR-AUC)

In [ ]:
from sklearn.metrics import make_scorer, average_precision_score
import time

print("="*80)
print("GRID SEARCH WITH FEATURE SELECTION (OPTIMIZED)")
print("="*80)

# ============================================================
# FEATURE SELECTION
# ============================================================
print("\n" + "="*60)
print("APPLYING FEATURE SELECTION")
print("="*60)

# Use the best threshold found earlier (from PR-AUC ranking)
GAIN_THRESHOLD = best_result_PRAUC['threshold']

print(f"Using threshold: > {GAIN_THRESHOLD}")
print(f"Original features: {len(X_train_imputed.columns)}")

# Select features based on threshold
selected_features = feature_importance[feature_importance['gain_score'] > GAIN_THRESHOLD]['feature'].tolist()

print(f"Selected features: {len(selected_features)}")
print(f"Feature reduction: {(1 - len(selected_features)/len(X_train_imputed.columns))*100:.1f}%")

# Filter datasets with selected features
X_train_selected = X_train_imputed[selected_features]
X_test_selected = X_test_imputed[selected_features]

# ============================================================
# PARAMETER GRID
# ============================================================
print("\n" + "="*60)
print("PARAMETER GRID")
print("="*60)

param_grid = {
    # EasyEnsemble parameters
    'n_estimators': [10, 25, 30],
    'sampling_strategy': ['auto', 1.0],

    # LightGBM parameters - kept most impactful combinations
    'estimator__n_estimators': [200, 500],
    'estimator__learning_rate': [0.05, 0.1],
    'estimator__max_depth': [-1], #default # add 7
    'estimator__num_leaves': [31] #default
}

print("\nOptimized Parameter Grid:")
for param, values in param_grid.items():
    print(f"  {param:<30}: {values}")

# Calculate total combinations
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)

n_folds = 3
total_fits = total_combinations * n_folds

print(f"\n{'='*60}")
print("COMPUTATIONAL COST")
print("="*60)
print(f"Total parameter combinations: {total_combinations:,}")
print(f"Cross-validation folds:       {n_folds}")
print(f"Total model fits:             {total_fits:,}")
print(f"Features used:                {len(selected_features)} / {len(X_train_imputed.columns)}")

# Create base model
base_model = EasyEnsembleClassifier(
    estimator=lgb.LGBMClassifier(
        random_state=random_seed,
        verbose=-1
    ),
    random_state=random_seed,
    n_jobs=-1
)

# Create GridSearchCV with PR-AUC scoring
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    scoring='average_precision',  # PR-AUC scoring
    n_jobs=-1,
    verbose=1,  # Minimal output
    return_train_score=False
)

# ============================================================
# RUN GRID SEARCH
# ============================================================
print("\n" + "="*80)
print("STARTING GRID SEARCH")
print("="*80)
print(f"Testing {total_combinations} parameter combinations with {len(selected_features)} features...\n")

start_time = time.time()

# Fit with SELECTED FEATURES
grid_search.fit(X_train_selected, y_train)

search_time = time.time() - start_time
search_minutes = search_time / 60

print(f"\n{'='*60}")
print("SEARCH COMPLETED")
print("="*60)
print(f"✓ Completed in {search_time:.2f}s ({search_minutes:.2f} min)")

# ============================================================
# BEST PARAMETERS
# ============================================================
print("\n" + "="*80)
print("BEST PARAMETERS (RANKED BY PR-AUC)")
print("="*80)

print(f"\n🏆 Best CV PR-AUC: {grid_search.best_score_:.4f}")
print(f"   Features used: {len(selected_features)}")

# Separate EasyEnsemble and LightGBM parameters
ee_params = {}
lgb_params = {}

for param, value in grid_search.best_params_.items():
    if param.startswith('estimator__'):
        lgb_params[param.replace('estimator__', '')] = value
    else:
        ee_params[param] = value

print("\nEasyEnsemble Parameters:")
for param, value in ee_params.items():
    print(f"  {param:<25}: {value}")

print("\nLightGBM Parameters:")
for param, value in lgb_params.items():
    print(f"  {param:<25}: {value}")

# Store parameters for later use
ee_n_estimators = ee_params.get('n_estimators', 10)
ee_sampling_strategy = ee_params.get('sampling_strategy', 'auto')
lgb_n_estimators = lgb_params.get('n_estimators', 200)
lgb_learning_rate = lgb_params.get('learning_rate', 0.1)
lgb_max_depth = lgb_params.get('max_depth', -1)
lgb_num_leaves = lgb_params.get('num_leaves', 31)

print("\n" + "="*80)
print("GRID SEARCH WITH FEATURE SELECTION COMPLETE")
print("="*80)

print(f"\n📊 Summary:")
print(f"  • Best PR-AUC:       {grid_search.best_score_:.4f}")
print(f"  • Search time:       {search_minutes:.1f} minutes")
print(f"  • Combinations:      {total_combinations}")
print(f"  • Features:          {len(selected_features)} / {len(X_train_imputed.columns)}")
print(f"  • Reduction:         {(1 - len(selected_features)/len(X_train_imputed.columns))*100:.1f}%")

print("\n✅ Ready to train final model with selected features and optimized parameters")

In [ ]:
from sklearn.metrics import fbeta_score, auc as calculate_auc
import time

print("\n" + "="*80)
print("FINAL MODEL: GRID SEARCH OPTIMIZED + FEATURE SELECTION")
print("="*80)

print("\nConfiguration:")
print("-" * 60)
print(f"Features:          {len(selected_features)} / {len(X_train_imputed.columns)} ({(1 - len(selected_features)/len(X_train_imputed.columns))*100:.1f}% reduction)")
print(f"Gain threshold:    > {GAIN_THRESHOLD}")

print("\nBest Parameters from Grid Search:")
print("-" * 60)
print("\nEasyEnsemble:")
print(f"  n_estimators:      {ee_n_estimators}")
print(f"  sampling_strategy: {ee_sampling_strategy}")

print("\nLightGBM:")
print(f"  n_estimators:      {lgb_n_estimators}")
print(f"  learning_rate:     {lgb_learning_rate}")
print(f"  max_depth:         {lgb_max_depth}")
print(f"  num_leaves:        {lgb_num_leaves}")

# ============================================================
# TRAIN FINAL MODEL
# ============================================================
print("\n" + "-" * 60)
print("Training final model...")
print("-" * 60)

start_time = time.time()

model_gridsearch_selected = EasyEnsembleClassifier(
    estimator=lgb.LGBMClassifier(
        random_state=random_seed,
        n_estimators=lgb_n_estimators,
        learning_rate=lgb_learning_rate,
        max_depth=lgb_max_depth,
        num_leaves=lgb_num_leaves,
        verbose=-1
    ),
    n_estimators=ee_n_estimators,
    sampling_strategy=ee_sampling_strategy,
    random_state=random_seed,
    n_jobs=-1
)

model_gridsearch_selected.fit(X_train_selected, y_train)
training_time = time.time() - start_time

# ============================================================
# PREDICTIONS
# ============================================================
y_pred_gs_selected = model_gridsearch_selected.predict(X_test_selected)
y_pred_proba_gs_selected = model_gridsearch_selected.predict_proba(X_test_selected)[:, 1]

# ============================================================
# CALCULATE METRICS
# ============================================================
# Calculate PR-AUC
# precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba_gs_selected)
# pr_auc_gs_selected = calculate_auc(recall_vals, precision_vals)
pr_auc_gs_selected = average_precision_score(y_test, y_pred_proba_gs_selected)

# Store results
results_gridsearch_selected = {
    'Method': 'GridSearch + Feature Selection',
    'Train_Size': len(y_train),
    'N_Features': len(selected_features),
    'Training_Time': training_time,
    'Accuracy': accuracy_score(y_test, y_pred_gs_selected),
    'Precision': precision_score(y_test, y_pred_gs_selected),
    'Recall': recall_score(y_test, y_pred_gs_selected),
    'F2_Score': fbeta_score(y_test, y_pred_gs_selected, beta=2),
    'ROC_AUC': roc_auc_score(y_test, y_pred_proba_gs_selected),
    'PR_AUC': pr_auc_gs_selected,
    'MCC': matthews_corrcoef(y_test, y_pred_gs_selected)
}

# ============================================================
# DISPLAY RESULTS
# ============================================================
print(f"\n✅ Model trained in {results_gridsearch_selected['Training_Time']:.2f}s")

print(f"\nTest Set Results:")
print(f"   Features:  {results_gridsearch_selected['N_Features']}")
print(f"   Accuracy:  {results_gridsearch_selected['Accuracy']:.4f}")
print(f"   Precision: {results_gridsearch_selected['Precision']:.4f}")
print(f"   Recall:    {results_gridsearch_selected['Recall']:.4f}")
print(f"   F2-Score:  {results_gridsearch_selected['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_gridsearch_selected['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_gridsearch_selected['PR_AUC']:.4f}")
print(f"   MCC:       {results_gridsearch_selected['MCC']:.4f}")

print("\nConfusion Matrix:")
cm_gs_selected = confusion_matrix(y_test, y_pred_gs_selected)
print(cm_gs_selected)

tn, fp, fn, tp = cm_gs_selected.ravel()
print(f"\n  TN: {tn:,} | FP: {fp:,}")
print(f"  FN: {fn:,} | TP: {tp:,}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gs_selected))

# Compare results
EasyEnsemble <br>
EasyEnsemble+FS <br>
EasyEnsemble+FS+Optuna <br>
EasyEnsemble+FS+GridSearch

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON: EasyEnsemble Variants")
print("="*80)

# ============================================================
# COLLECT RESULTS FOR SPECIFIC MODELS
# ============================================================

comparison_results = []

# 1. EasyEnsemble (All Features)
comparison_results.append({
    'Model': 'EasyEnsemble',
    'N_Features': results_EE['N_Features'],
    'Training_Time': results_EE['Training_Time'],
    'Accuracy': results_EE['Accuracy'],
    'Precision': results_EE['Precision'],
    'Recall': results_EE['Recall'],
    'F2_Score': results_EE['F2_Score'],
    'ROC_AUC': results_EE['ROC_AUC'],
    'PR_AUC': results_EE['PR_AUC'],
    'MCC': results_EE['MCC']
})

# 2. EasyEnsemble + Feature Selection
comparison_results.append({
    'Model': 'EasyEnsemble + FS',
    'N_Features': results_selected_prauc['N_Features'],
    'Training_Time': results_selected_prauc['Training_Time'],
    'Accuracy': results_selected_prauc['Accuracy'],
    'Precision': results_selected_prauc['Precision'],
    'Recall': results_selected_prauc['Recall'],
    'F2_Score': results_selected_prauc['F2_Score'],
    'ROC_AUC': results_selected_prauc['ROC_AUC'],
    'PR_AUC': results_selected_prauc['PR_AUC'],
    'MCC': results_selected_prauc['MCC']
    })

# 3. EasyEnsemble + FS + Optuna
if 'results_optuna_selected' in locals():
    comparison_results.append({
        'Model': 'EasyEnsemble + FS + Optuna',
        'N_Features': results_optuna_selected['N_Features'],
        'Training_Time': results_optuna_selected['Training_Time'],
        'Accuracy': results_optuna_selected['Accuracy'],
        'Precision': results_optuna_selected['Precision'],
        'Recall': results_optuna_selected['Recall'],
        'F2_Score': results_optuna_selected['F2_Score'],
        'ROC_AUC': results_optuna_selected['ROC_AUC'],
        'PR_AUC': results_optuna_selected['PR_AUC'],
        'MCC': results_optuna_selected['MCC']
    })

# 4. EasyEnsemble + FS + GridSearch
if 'results_gridsearch_selected' in locals():
    comparison_results.append({
        'Model': 'EasyEnsemble + FS + GridSearch',
        'N_Features': results_gridsearch_selected['N_Features'],
        'Training_Time': results_gridsearch_selected['Training_Time'],
        'Accuracy': results_gridsearch_selected['Accuracy'],
        'Precision': results_gridsearch_selected['Precision'],
        'Recall': results_gridsearch_selected['Recall'],
        'F2_Score': results_gridsearch_selected['F2_Score'],
        'ROC_AUC': results_gridsearch_selected['ROC_AUC'],
        'PR_AUC': results_gridsearch_selected['PR_AUC'],
        'MCC': results_gridsearch_selected['MCC']
    })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_results)

# ============================================================
# MULTI-CRITERIA RANKING: PR-AUC > F2 > MCC
# ============================================================
print("\n" + "="*80)
print("RANKING BY: PR-AUC (Primary) > F2-Score (Secondary) > MCC (Tertiary)")
print("="*80)

# Sort by PR-AUC (descending), then F2 (descending), then MCC (descending)
comparison_df_ranked = comparison_df.sort_values(
    by=['PR_AUC', 'F2_Score', 'MCC'],
    ascending=[False, False, False]
).reset_index(drop=True)

comparison_df_ranked.insert(0, 'Rank', range(1, len(comparison_df_ranked) + 1))

print("\n" + comparison_df_ranked.to_string(index=False))

# ============================================================
# BEST MODEL SUMMARY
# ============================================================
best_model = comparison_df_ranked.iloc[0]

print("\n" + "="*80)
print("🏆 WINNER: " + best_model['Model'])
print("="*80)
print(f"\nPrimary Metric (PR-AUC):      {best_model['PR_AUC']:.4f}")
print(f"Secondary Metric (F2-Score):  {best_model['F2_Score']:.4f}")
print(f"Tertiary Metric (MCC):        {best_model['MCC']:.4f}")
print(f"\nOther Metrics:")
print(f"  Accuracy:      {best_model['Accuracy']:.4f}")
print(f"  Precision:     {best_model['Precision']:.4f}")
print(f"  Recall:        {best_model['Recall']:.4f}")
print(f"  ROC-AUC:       {best_model['ROC_AUC']:.4f}")
print(f"\nModel Configuration:")
print(f"  Features:      {int(best_model['N_Features'])}")
print(f"  Training Time: {best_model['Training_Time']:.2f}s")


# ============================================================
# FINAL RECOMMENDATION
# ============================================================
print("\n" + "="*80)
print("FINAL RECOMMENDATION")
print("="*80)

best = comparison_df_ranked.iloc[0]
second = comparison_df_ranked.iloc[1] if len(comparison_df_ranked) > 1 else None

print(f"\n🏆 Best Model: {best['Model']}")
print(f"   PR-AUC:   {best['PR_AUC']:.4f}")
print(f"   F2-Score: {best['F2_Score']:.4f}")
print(f"   MCC:      {best['MCC']:.4f}")
print(f"   Features: {int(best['N_Features'])}")

if second is not None:
    pr_auc_diff = best['PR_AUC'] - second['PR_AUC']
    f2_diff = best['F2_Score'] - second['F2_Score']

    if pr_auc_diff < 0.001:  # Very close
        print(f"\n⚠️  Note: {best['Model']} and {second['Model']} have very similar performance")
        print(f"   Consider {second['Model']} if:")
        if second['Training_Time'] < best['Training_Time']:
            print(f"   - Faster training is important ({second['Training_Time']:.2f}s vs {best['Training_Time']:.2f}s)")
        if second['N_Features'] < best['N_Features']:
            print(f"   - Simpler model is preferred ({int(second['N_Features'])} vs {int(best['N_Features'])} features)")
    else:
        print(f"\n✅ {best['Model']} is clearly superior")
        print(f"   Outperforms {second['Model']} by:")
        print(f"   - PR-AUC: +{pr_auc_diff:.4f}")
        print(f"   - F2-Score: +{f2_diff:.4f}")

print("\n" + "="*80)

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# BEST MODEL VISUALIZATION
# ============================================================
print("\n" + "="*80)
print("BEST MODEL VISUALIZATION")
print("="*80)

metrics = ['PR-AUC', 'F2-Score', 'MCC', 'Recall']
values = [
    best_model['PR_AUC'],
    best_model['F2_Score'],
    best_model['MCC'],
    best_model['Recall']
]

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
bars = ax.bar(metrics, values, color=colors, edgecolor='black', linewidth=1.5, width=0.6)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f'{val:.4f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Score', fontsize=14, fontweight='bold')
ax.set_title(f"🏆 {best_model['Model']} Performance", fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('lgbm_best_model_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: lgbm_best_model_metrics.png")

### Save the best model

In [ ]:
from joblib import dump
import os

print("\n" + "="*80)
print("SAVING BEST MODEL")
print("="*80)

# Create models directory if it doesn't exist
os.makedirs("models", exist_ok=True)

# Get the best model from comparison
best_model_name = comparison_df_ranked.iloc[0]['Model']
best_pr_auc = comparison_df_ranked.iloc[0]['PR_AUC']
best_f2 = comparison_df_ranked.iloc[0]['F2_Score']
best_mcc = comparison_df_ranked.iloc[0]['MCC']
best_n_features = int(comparison_df_ranked.iloc[0]['N_Features'])

print(f"\nBest Model: {best_model_name}")
print(f"  PR-AUC:   {best_pr_auc:.4f}")
print(f"  F2-Score: {best_f2:.4f}")
print(f"  MCC:      {best_mcc:.4f}")
print(f"  Features: {best_n_features}")

# Determine which model to save
if best_model_name == 'EasyEnsemble':
    model_to_save = easy_ensemble
    selected_features_to_save = None
    filename = "models/easyensemble_baseline.pkl"
    features_filename = None

elif best_model_name == 'EasyEnsemble + FS':
    model_to_save = model_selected_prauc
    selected_features_to_save = selected_features_prauc
    filename = "models/easyensemble_fs_prauc.pkl"
    features_filename = "models/easyensemble_fs_prauc_features.pkl"

elif best_model_name == 'EasyEnsemble + FS + Optuna':
    model_to_save = model_optuna_selected
    selected_features_to_save = selected_features
    filename = "models/easyensemble_fs_optuna.pkl"
    features_filename = "models/easyensemble_fs_optuna_features.pkl"

elif best_model_name == 'EasyEnsemble + FS + GridSearch':
    model_to_save = model_gridsearch_selected
    selected_features_to_save = selected_features
    filename = "models/easyensemble_fs_gridsearch.pkl"
    features_filename = "models/easyensemble_fs_gridsearch_features.pkl"

# Save the model
print(f"\nSaving model to: {filename}")
dump(model_to_save, filename)
print(f"✓ Model saved successfully")


# Big *Dataset*

### model run

In [ ]:
# BIG DATA
print("\n" + "="*80)
print("RUNNING ON BIG DATA WITH SAME PARAMETERS")
print("="*80)

# parameter check
print("\n" + "-"*60)
print("PARAMETERS USED (from Small Data Best Model):")
print("-"*60)
print(f"GAIN_THRESHOLD:       {GAIN_THRESHOLD}")
print(f"selected_features:    {len(selected_features)} features")
print(f"\nEasyEnsemble:")
print(f"  n_estimators:       {ee_n_estimators}")
print(f"  sampling_strategy:  {ee_sampling_strategy}")
print(f"\nLightGBM:")
print(f"  n_estimators:       {lgb_n_estimators}")
print(f"  learning_rate:      {lgb_learning_rate}")
print(f"  max_depth:          {lgb_max_depth}")
print(f"  num_leaves:         {lgb_num_leaves}")

# Big data
DATA_PATH_BIG = '/content/datafile.csv'  # ← Big data 경로
df_big = pd.read_csv(DATA_PATH_BIG)
df_big = df_big.drop(['ID_code', 'var_45'], axis=1)

X_big = df_big.drop("target", axis=1)
y_big = df_big["target"]

print(f"\nBig data shape: {df_big.shape}")

# Train-test split
X_train_big, X_test_big, y_train_big, y_test_big = train_test_split(
    X_big, y_big,
    test_size=0.2,
    random_state=random_seed,
    stratify=y_big
)

# Imputation
imputer_big = SimpleImputer(strategy="mean")
X_train_big_imp = pd.DataFrame(
    imputer_big.fit_transform(X_train_big),
    columns=X_train_big.columns
)
X_test_big_imp = pd.DataFrame(
    imputer_big.transform(X_test_big),
    columns=X_test_big.columns
)

# Feature selection (same selected_features)
X_train_big_selected = X_train_big_imp[selected_features]
X_test_big_selected = X_test_big_imp[selected_features]

print(f"Using {len(selected_features)} features from small data")

# Train
model_big = EasyEnsembleClassifier(
    estimator=lgb.LGBMClassifier(
        random_state=random_seed,
        n_estimators=lgb_n_estimators,
        learning_rate=lgb_learning_rate,
        max_depth=lgb_max_depth,
        num_leaves=lgb_num_leaves,
        verbose=-1
    ),
    n_estimators=ee_n_estimators,
    sampling_strategy=ee_sampling_strategy,
    random_state=random_seed,
    n_jobs=-1
)

model_big.fit(X_train_big_selected, y_train_big)
training_time_big = time.time() - start_time
print(f"Trained in {training_time_big:.2f}s")

# Predict & Evaluate
y_pred_big = model_big.predict(X_test_big_selected)
y_proba_big = model_big.predict_proba(X_test_big_selected)[:, 1]

# RESULTS - SAME FORMAT AS SMALL DATA
print("\n" + "="*80)
print("🏆 BIG DATA RESULTS (Same Parameters as Small Data Best Model)")
print("="*80)

print(f"\nPrimary Metric (PR-AUC):      {average_precision_score(y_test_big, y_proba_big):.4f}")
print(f"Secondary Metric (F2-Score):  {fbeta_score(y_test_big, y_pred_big, beta=2):.4f}")
print(f"Tertiary Metric (MCC):        {matthews_corrcoef(y_test_big, y_pred_big):.4f}")

print(f"\nOther Metrics:")
print(f"  Accuracy:      {accuracy_score(y_test_big, y_pred_big):.4f}")
print(f"  Precision:     {precision_score(y_test_big, y_pred_big):.4f}")
print(f"  Recall:        {recall_score(y_test_big, y_pred_big):.4f}")
print(f"  ROC-AUC:       {roc_auc_score(y_test_big, y_proba_big):.4f}")

print(f"\nModel Configuration:")
print(f"  Features:      {len(selected_features)}")
print(f"  Training Time: {training_time_big:.2f}s")

print("\nConfusion Matrix:")
cm_big = confusion_matrix(y_test_big, y_pred_big)
print(cm_big)

tn, fp, fn, tp = cm_big.ravel()
print(f"\n  TN: {tn:,} | FP: {fp:,}")
print(f"  FN: {fn:,} | TP: {tp:,}")

# COMPARISON: SMALL vs BIG
print("\n" + "="*80)
print("COMPARISON: SMALL DATA vs BIG DATA")
print("="*80)

print(f"\n{'Metric':<15} {'Small Data':<15} {'Big Data':<15}")
print("="*45)
print(f"{'PR-AUC':<15} {best_model['PR_AUC']:<15.4f} {average_precision_score(y_test_big, y_proba_big):<15.4f}")
print(f"{'F2-Score':<15} {best_model['F2_Score']:<15.4f} {fbeta_score(y_test_big, y_pred_big, beta=2):<15.4f}")
print(f"{'MCC':<15} {best_model['MCC']:<15.4f} {matthews_corrcoef(y_test_big, y_pred_big):<15.4f}")
print(f"{'Recall':<15} {best_model['Recall']:<15.4f} {recall_score(y_test_big, y_pred_big):<15.4f}")

## Visualisation

### Colorful Visusalisation

In [ ]:
# ============================================================================
# VISUALIZATION - 개별 파일 저장
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

from sklearn.metrics import precision_recall_curve, roc_curve

results_big = {
    'PR_AUC': average_precision_score(y_test_big, y_proba_big),
    'F2_Score': fbeta_score(y_test_big, y_pred_big, beta=2),
    'MCC': matthews_corrcoef(y_test_big, y_pred_big),
    'Recall': recall_score(y_test_big, y_pred_big),
    'Precision': precision_score(y_test_big, y_pred_big),
    'Accuracy': accuracy_score(y_test_big, y_pred_big),
    'ROC_AUC': roc_auc_score(y_test_big, y_proba_big)
}

# ============================================================================
# 1. PR-AUC Curve
# ============================================================================
fig1, ax1 = plt.subplots(figsize=(10, 8))

precision_vals, recall_vals, _ = precision_recall_curve(y_test_big, y_proba_big)

ax1.plot(recall_vals, precision_vals, color='#2ecc71', linewidth=2.5,
         label=f'LGBM (PR-AUC = {results_big["PR_AUC"]:.4f})')
ax1.fill_between(recall_vals, precision_vals, alpha=0.3, color='#2ecc71')

baseline = y_test_big.mean()
ax1.axhline(y=baseline, color='red', linestyle='--', linewidth=1.5,
            label=f'Baseline = {baseline:.4f}')

ax1.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax1.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax1.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig1_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig1_pr_curve.png")

# ============================================================================
# 2. ROC Curve
# ============================================================================
fig2, ax2 = plt.subplots(figsize=(10, 8))

fpr, tpr, _ = roc_curve(y_test_big, y_proba_big)

ax2.plot(fpr, tpr, color='#3498db', linewidth=2.5,
         label=f'LGBM (ROC-AUC = {results_big["ROC_AUC"]:.4f})')
ax2.fill_between(fpr, tpr, alpha=0.3, color='#3498db')
ax2.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=1.5,
         label='Baseline (Random)')

ax2.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax2.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax2.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig2_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig2_roc_curve.png")

# ============================================================================
# 3. Key Metrics Bar Chart
# ============================================================================
fig3, ax3 = plt.subplots(figsize=(10, 8))

metrics = ['PR-AUC\n(Primary)', 'F2-Score\n(Secondary)', 'MCC\n(Tertiary)', 'Recall']
values = [results_big['PR_AUC'], results_big['F2_Score'],
          results_big['MCC'], results_big['Recall']]
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']

bars = ax3.bar(metrics, values, color=colors, edgecolor='black', linewidth=1.5, width=0.6)

for bar, val in zip(bars, values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax3.set_ylabel('Score', fontsize=12, fontweight='bold')
ax3.set_title('LGBM Best Model - Key Metrics', fontsize=14, fontweight='bold')
ax3.set_ylim(0, 1.0)
ax3.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('fig3_key_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig3_key_metrics.png")

# ============================================================================
# 4. Small vs Big Data Comparison
# ============================================================================
fig4, ax4 = plt.subplots(figsize=(10, 8))

metrics_compare = ['PR-AUC', 'F2-Score', 'MCC', 'Recall']
small_values = [best_model['PR_AUC'], best_model['F2_Score'],
                best_model['MCC'], best_model['Recall']]
big_values = [results_big['PR_AUC'], results_big['F2_Score'],
              results_big['MCC'], results_big['Recall']]

x = np.arange(len(metrics_compare))
width = 0.35

bars1 = ax4.bar(x - width/2, small_values, width, label='Small Data',
                color='#3498db', edgecolor='black', linewidth=1.5)
bars2 = ax4.bar(x + width/2, big_values, width, label='Big Data',
                color='#e74c3c', edgecolor='black', linewidth=1.5)

for bar in bars1:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
ax4.set_title('Small Data vs Big Data Comparison', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics_compare)
ax4.legend(loc='upper right', fontsize=10)
ax4.set_ylim(0, 1.0)
ax4.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('fig4_small_vs_big_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig4_small_vs_big_comparison.png")

# ============================================================================
# 5. Confusion Matrix Heatmap
# ============================================================================
fig5, ax5 = plt.subplots(figsize=(8, 6))

im = ax5.imshow(cm_big, cmap='Blues')

for i in range(2):
    for j in range(2):
        color = 'white' if cm_big[i, j] > cm_big.max()/2 else 'black'
        ax5.text(j, i, f'{cm_big[i, j]:,}', ha='center', va='center',
                 fontsize=20, fontweight='bold', color=color)

ax5.set_xticks([0, 1])
ax5.set_yticks([0, 1])
ax5.set_xticklabels(['Predicted 0', 'Predicted 1'], fontsize=12)
ax5.set_yticklabels(['Actual 0', 'Actual 1'], fontsize=12)
ax5.set_title('Confusion Matrix - LGBM Big Data', fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax5)
plt.tight_layout()
plt.savefig('fig5_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig5_confusion_matrix.png")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("VISUALIZATION COMPLETE!")
print("="*80)
print("\nSaved Files:")
print("  1. fig1_pr_curve.png")
print("  2. fig2_roc_curve.png")
print("  3. fig3_key_metrics.png")
print("  4. fig4_small_vs_big_comparison.png")
print("  5. fig5_confusion_matrix.png")

In [ ]:
# ============================================================================
# 1b. PR-AUC Curve (WITHOUT Baseline)
# ============================================================================
fig1b, ax1b = plt.subplots(figsize=(10, 8))

ax1b.plot(recall_vals, precision_vals, color='#2ecc71', linewidth=2.5,
         label=f'LGBM (PR-AUC = {results_big["PR_AUC"]:.4f})')
ax1b.fill_between(recall_vals, precision_vals, alpha=0.3, color='#2ecc71')

ax1b.set_xlabel('Recall', fontsize=12, fontweight='bold')
ax1b.set_ylabel('Precision', fontsize=12, fontweight='bold')
ax1b.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax1b.legend(loc='upper right', fontsize=10)
ax1b.grid(True, alpha=0.3)
ax1b.set_xlim([0, 1])
ax1b.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig1_pr_curve_no_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig1_pr_curve_no_baseline.png")

# ============================================================================
# 2b. ROC Curve (WITHOUT Baseline)
# ============================================================================
fig2b, ax2b = plt.subplots(figsize=(10, 8))

ax2b.plot(fpr, tpr, color='#3498db', linewidth=2.5,
         label=f'LGBM (ROC-AUC = {results_big["ROC_AUC"]:.4f})')
ax2b.fill_between(fpr, tpr, alpha=0.3, color='#3498db')

ax2b.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax2b.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax2b.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2b.legend(loc='lower right', fontsize=10)
ax2b.grid(True, alpha=0.3)
ax2b.set_xlim([0, 1])
ax2b.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig2_roc_curve_no_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig2_roc_curve_no_baseline.png")

### Grey Visualisation

In [ ]:
# ============================================================================
# VISUALIZATION - TECHNICAL REPORT STYLE
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS (Technical Report Style)")
print("="*80)

from sklearn.metrics import precision_recall_curve, roc_curve

COLOR_PRIMARY = '#2c3e50'
COLOR_SECONDARY = '#7f8c8d'
COLOR_ACCENT = '#34495e'
COLOR_LIGHT = '#bdc3c7'
COLOR_BASELINE = '#e74c3c'

# Set global style
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

results_big = {
    'PR_AUC': average_precision_score(y_test_big, y_proba_big),
    'F2_Score': fbeta_score(y_test_big, y_pred_big, beta=2),
    'MCC': matthews_corrcoef(y_test_big, y_pred_big),
    'Recall': recall_score(y_test_big, y_pred_big),
    'Precision': precision_score(y_test_big, y_pred_big),
    'Accuracy': accuracy_score(y_test_big, y_pred_big),
    'ROC_AUC': roc_auc_score(y_test_big, y_proba_big)
}

# ============================================================================
# 1. PR-AUC Curve (WITH Baseline)
# ============================================================================
fig1, ax1 = plt.subplots(figsize=(10, 8))

precision_vals, recall_vals, _ = precision_recall_curve(y_test_big, y_proba_big)

ax1.plot(recall_vals, precision_vals, color=COLOR_PRIMARY, linewidth=2.5,
         label=f'LGBM (PR-AUC = {results_big["PR_AUC"]:.4f})')
ax1.fill_between(recall_vals, precision_vals, alpha=0.15, color=COLOR_PRIMARY)

baseline = y_test_big.mean()
ax1.axhline(y=baseline, color=COLOR_BASELINE, linestyle='--', linewidth=1.5,
            label=f'Baseline = {baseline:.4f}')

ax1.set_xlabel('Recall', fontsize=12)
ax1.set_ylabel('Precision', fontsize=12)
ax1.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3, linestyle='-', color=COLOR_LIGHT)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig1_pr_curve_with_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig1_pr_curve_with_baseline.png")

# ============================================================================
# 1b. PR-AUC Curve (WITHOUT Baseline)
# ============================================================================
fig1b, ax1b = plt.subplots(figsize=(10, 8))

ax1b.plot(recall_vals, precision_vals, color=COLOR_PRIMARY, linewidth=2.5,
         label=f'LGBM (PR-AUC = {results_big["PR_AUC"]:.4f})')
ax1b.fill_between(recall_vals, precision_vals, alpha=0.15, color=COLOR_PRIMARY)

ax1b.set_xlabel('Recall', fontsize=12)
ax1b.set_ylabel('Precision', fontsize=12)
ax1b.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax1b.legend(loc='upper right', fontsize=10)
ax1b.grid(True, alpha=0.3, linestyle='-', color=COLOR_LIGHT)
ax1b.set_xlim([0, 1])
ax1b.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig1_pr_curve_no_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig1_pr_curve_no_baseline.png")

# ============================================================================
# 2. ROC Curve (WITH Baseline)
# ============================================================================
fig2, ax2 = plt.subplots(figsize=(10, 8))

fpr, tpr, _ = roc_curve(y_test_big, y_proba_big)

ax2.plot(fpr, tpr, color=COLOR_PRIMARY, linewidth=2.5,
         label=f'LGBM (ROC-AUC = {results_big["ROC_AUC"]:.4f})')
ax2.fill_between(fpr, tpr, alpha=0.15, color=COLOR_PRIMARY)
ax2.plot([0, 1], [0, 1], color=COLOR_BASELINE, linestyle='--', linewidth=1.5,
         label='Baseline (Random)')

ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='-', color=COLOR_LIGHT)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig2_roc_curve_with_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig2_roc_curve_with_baseline.png")

# ============================================================================
# 2b. ROC Curve (WITHOUT Baseline)
# ============================================================================
fig2b, ax2b = plt.subplots(figsize=(10, 8))

ax2b.plot(fpr, tpr, color=COLOR_PRIMARY, linewidth=2.5,
         label=f'LGBM (ROC-AUC = {results_big["ROC_AUC"]:.4f})')
ax2b.fill_between(fpr, tpr, alpha=0.15, color=COLOR_PRIMARY)

ax2b.set_xlabel('False Positive Rate', fontsize=12)
ax2b.set_ylabel('True Positive Rate', fontsize=12)
ax2b.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2b.legend(loc='lower right', fontsize=10)
ax2b.grid(True, alpha=0.3, linestyle='-', color=COLOR_LIGHT)
ax2b.set_xlim([0, 1])
ax2b.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig2_roc_curve_no_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig2_roc_curve_no_baseline.png")

# ============================================================================
# 3. Key Metrics Bar Chart
# ============================================================================
fig3, ax3 = plt.subplots(figsize=(10, 8))

metrics = ['PR-AUC\n(Primary)', 'F2-Score\n(Secondary)', 'MCC\n(Tertiary)', 'Recall']
values = [results_big['PR_AUC'], results_big['F2_Score'],
          results_big['MCC'], results_big['Recall']]

# 단색 그라데이션
colors_gray = [COLOR_PRIMARY, COLOR_ACCENT, COLOR_SECONDARY, COLOR_LIGHT]

bars = ax3.bar(metrics, values, color=colors_gray, edgecolor='black', linewidth=1, width=0.6)

for bar, val in zip(bars, values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax3.set_ylabel('Score', fontsize=12)
ax3.set_title('LGBM Best Model - Key Metrics', fontsize=14, fontweight='bold')
ax3.set_ylim(0, 1.0)
ax3.grid(axis='y', alpha=0.3, linestyle='-', color=COLOR_LIGHT)

plt.tight_layout()
plt.savefig('fig3_key_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig3_key_metrics.png")

# ============================================================================
# 4. Small vs Big Data Comparison
# ============================================================================
fig4, ax4 = plt.subplots(figsize=(10, 8))

metrics_compare = ['PR-AUC', 'F2-Score', 'MCC', 'Recall']
small_values = [best_model['PR_AUC'], best_model['F2_Score'],
                best_model['MCC'], best_model['Recall']]
big_values = [results_big['PR_AUC'], results_big['F2_Score'],
              results_big['MCC'], results_big['Recall']]

x = np.arange(len(metrics_compare))
width = 0.35

bars1 = ax4.bar(x - width/2, small_values, width, label='Small Data',
                color=COLOR_SECONDARY, edgecolor='black', linewidth=1)
bars2 = ax4.bar(x + width/2, big_values, width, label='Big Data',
                color=COLOR_PRIMARY, edgecolor='black', linewidth=1)

for bar in bars1:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax4.set_ylabel('Score', fontsize=12)
ax4.set_title('Small Data vs Big Data Comparison', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics_compare)
ax4.legend(loc='upper right', fontsize=10)
ax4.set_ylim(0, 1.0)
ax4.grid(axis='y', alpha=0.3, linestyle='-', color=COLOR_LIGHT)

plt.tight_layout()
plt.savefig('fig4_small_vs_big_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig4_small_vs_big_comparison.png")

# ============================================================================
# 5. Confusion Matrix Heatmap
# ============================================================================
fig5, ax5 = plt.subplots(figsize=(8, 6))

# 회색 계열 colormap
im = ax5.imshow(cm_big, cmap='Greys')

for i in range(2):
    for j in range(2):
        color = 'white' if cm_big[i, j] > cm_big.max()/2 else 'black'
        ax5.text(j, i, f'{cm_big[i, j]:,}', ha='center', va='center',
                 fontsize=20, fontweight='bold', color=color)

ax5.set_xticks([0, 1])
ax5.set_yticks([0, 1])
ax5.set_xticklabels(['Predicted 0', 'Predicted 1'], fontsize=12)
ax5.set_yticklabels(['Actual 0', 'Actual 1'], fontsize=12)
ax5.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax5)
plt.tight_layout()
plt.savefig('fig5_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig5_confusion_matrix.png")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("VISUALIZATION COMPLETE!")
print("="*80)
print("\nSaved Files:")
print("  1. fig1_pr_curve_with_baseline.png")
print("  2. fig1_pr_curve_no_baseline.png")
print("  3. fig2_roc_curve_with_baseline.png")
print("  4. fig2_roc_curve_no_baseline.png")
print("  5. fig3_key_metrics.png")
print("  6. fig4_small_vs_big_comparison.png")
print("  7. fig5_confusion_matrix.png")

## SHAP & Feature Importance

### Colorful Visualisation

In [ ]:
# ============================================================================
# FEATURE IMPORTANCE & SHAP ANALYSIS (BIG DATA)
# ============================================================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE & SHAP ANALYSIS")
print("="*80)

import shap

# ============================================================================
# 1. GLOBAL FEATURE IMPORTANCE (Gain-based)
# ============================================================================
print("\n" + "-"*60)
print("1. GLOBAL FEATURE IMPORTANCE (Gain-based)")
print("-"*60)

# Extract feature importances from all estimators
feature_importances_list = []
for pipeline in model_big.estimators_:
    classifier = pipeline.steps[-1][1]  # LGBMClassifier
    feature_importances_list.append(classifier.feature_importances_)

avg_importance = np.mean(feature_importances_list, axis=0)

# Create DataFrame
feature_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': avg_importance
}).sort_values('importance', ascending=False)

print(f"\nTop 20 Features by Gain Score:")
print(feature_importance_df.head(20).to_string(index=False))

# Save to CSV
feature_importance_df.to_csv('feature_importance_bigdata.csv', index=False)
print("\n✓ Saved: feature_importance_bigdata.csv")

# ============================================================================
# 2. SHAP ANALYSIS
# ============================================================================
print("\n" + "-"*60)
print("2. SHAP ANALYSIS")
print("-"*60)

# Use first estimator from EasyEnsemble
lgbm_single = model_big.estimators_[0].steps[-1][1]

# Create SHAP explainer
print("Creating SHAP TreeExplainer...")
explainer = shap.TreeExplainer(lgbm_single)

# Sample data for SHAP
sample_size = min(1000, len(X_test_big_selected))
np.random.seed(random_seed)
sample_idx = np.random.choice(len(X_test_big_selected), sample_size, replace=False)
X_sample = X_test_big_selected.iloc[sample_idx]

print(f"Calculating SHAP values for {sample_size} samples...")
shap_values = explainer.shap_values(X_sample)

# For binary classification
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

print("✓ SHAP values calculated")

# Mean absolute SHAP values
mean_shap = np.abs(shap_values_class1).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'feature': selected_features,
    'mean_shap': mean_shap
}).sort_values('mean_shap', ascending=False)

print(f"\nTop 20 Features by Mean |SHAP|:")
print(shap_importance_df.head(20).to_string(index=False))

# Save to CSV
shap_importance_df.to_csv('shap_importance_bigdata.csv', index=False)
print("\n✓ Saved: shap_importance_bigdata.csv")

# ============================================================================
# 3. FEATURE IMPORTANCE BAR PLOT (Top 20)
# ============================================================================
print("\n" + "-"*60)
print("3. FEATURE IMPORTANCE VISUALIZATION")
print("-"*60)

fig3, ax3 = plt.subplots(figsize=(10, 10))

top20_importance = feature_importance_df.head(20)
colors = plt.cm.Greens(np.linspace(0.4, 0.9, 20))[::-1]

bars = ax3.barh(range(len(top20_importance)), top20_importance['importance'].values,
                color=colors, edgecolor='black', linewidth=0.5)
ax3.set_yticks(range(len(top20_importance)))
ax3.set_yticklabels(top20_importance['feature'].values)
ax3.invert_yaxis()
ax3.set_xlabel('Gain Score', fontsize=12, fontweight='bold')
ax3.set_title('Global Feature Importance (Top 20)\nGain-based', fontsize=14, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig6_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig6_feature_importance.png")

# ============================================================================
# 4. SHAP BAR PLOT (Top 20)
# ============================================================================
fig4, ax4 = plt.subplots(figsize=(10, 10))

top20_shap = shap_importance_df.head(20)
colors_shap = plt.cm.Reds(np.linspace(0.4, 0.9, 20))[::-1]

bars2 = ax4.barh(range(len(top20_shap)), top20_shap['mean_shap'].values,
                 color=colors_shap, edgecolor='black', linewidth=0.5)
ax4.set_yticks(range(len(top20_shap)))
ax4.set_yticklabels(top20_shap['feature'].values)
ax4.invert_yaxis()
ax4.set_xlabel('Mean |SHAP Value|', fontsize=12, fontweight='bold')
ax4.set_title('SHAP Feature Importance (Top 20)\nMean Absolute SHAP', fontsize=14, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig7_shap_importance_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig7_shap_importance_bar.png")

# ============================================================================
# 5. SHAP SUMMARY PLOT (Beeswarm)
# ============================================================================
print("\n" + "-"*60)
print("5. SHAP SUMMARY PLOT (Beeswarm)")
print("-"*60)

plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_class1, X_sample, plot_type="dot",
                  max_display=20, show=False)
plt.title('SHAP Summary Plot (Top 20 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig8_shap_summary_beeswarm.png")

# ============================================================================
# 6. SHAP BAR PLOT (Built-in)
# ============================================================================
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_class1, X_sample, plot_type="bar",
                  max_display=20, show=False)
plt.title('SHAP Feature Importance (Bar)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_shap_bar_builtin.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig9_shap_bar_builtin.png")

# ============================================================================
# 7. GAIN vs SHAP COMPARISON (Top 15)
# ============================================================================
print("\n" + "-"*60)
print("7. GAIN vs SHAP COMPARISON")
print("-"*60)

fig7, ax7 = plt.subplots(figsize=(12, 8))

top15_features = feature_importance_df.head(15)['feature'].tolist()

# Get values for same features
shap_vals_top15 = []
for f in top15_features:
    val = shap_importance_df[shap_importance_df['feature'] == f]['mean_shap'].values
    shap_vals_top15.append(val[0] if len(val) > 0 else 0)

importance_vals_top15 = feature_importance_df.head(15)['importance'].values

# Normalize for comparison
importance_norm = importance_vals_top15 / importance_vals_top15.max()
shap_norm = np.array(shap_vals_top15) / max(shap_vals_top15)

x = np.arange(len(top15_features))
width = 0.35

bars1 = ax7.bar(x - width/2, importance_norm, width, label='Gain (normalized)',
                color='#2ecc71', edgecolor='black', linewidth=0.5)
bars2 = ax7.bar(x + width/2, shap_norm, width, label='SHAP (normalized)',
                color='#e74c3c', edgecolor='black', linewidth=0.5)

ax7.set_xticks(x)
ax7.set_xticklabels(top15_features, rotation=45, ha='right', fontsize=9)
ax7.set_ylabel('Normalized Importance', fontsize=12, fontweight='bold')
ax7.set_title('Gain vs SHAP Importance Comparison (Top 15)', fontsize=14, fontweight='bold')
ax7.legend(loc='upper right')
ax7.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig10_gain_vs_shap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig10_gain_vs_shap_comparison.png")

# ============================================================================
# 8. RANKING CONSISTENCY (Spearman Correlation)
# ============================================================================
print("\n" + "-"*60)
print("8. RANKING CONSISTENCY")
print("-"*60)

fig8, ax8 = plt.subplots(figsize=(10, 8))

# Get ranks
gain_rank = feature_importance_df.reset_index(drop=True).copy()
gain_rank['gain_rank'] = range(1, len(gain_rank) + 1)

shap_rank = shap_importance_df.reset_index(drop=True).copy()
shap_rank['shap_rank'] = range(1, len(shap_rank) + 1)

# Merge
rank_comparison = gain_rank.merge(shap_rank[['feature', 'shap_rank']], on='feature')
rank_comparison = rank_comparison.head(30)

ax8.scatter(rank_comparison['gain_rank'], rank_comparison['shap_rank'],
            s=100, c='#3498db', edgecolor='black', alpha=0.7)

# Perfect correlation line
max_rank = max(rank_comparison['gain_rank'].max(), rank_comparison['shap_rank'].max())
ax8.plot([1, max_rank], [1, max_rank], 'r--', linewidth=2, label='Perfect Agreement')

# Annotate top 10
for idx, row in rank_comparison.head(10).iterrows():
    ax8.annotate(row['feature'], (row['gain_rank'], row['shap_rank']),
                 fontsize=8, ha='left', va='bottom')

ax8.set_xlabel('Gain Importance Rank', fontsize=12, fontweight='bold')
ax8.set_ylabel('SHAP Importance Rank', fontsize=12, fontweight='bold')
ax8.set_title('Ranking Consistency: Gain vs SHAP', fontsize=14, fontweight='bold')
ax8.legend(loc='lower right')
ax8.grid(True, alpha=0.3)

# Spearman correlation
from scipy.stats import spearmanr
corr, pval = spearmanr(rank_comparison['gain_rank'], rank_comparison['shap_rank'])
ax8.text(0.05, 0.95, f'Spearman ρ = {corr:.3f}\np-value = {pval:.4f}',
         transform=ax8.transAxes, fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('fig11_ranking_consistency.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: fig11_ranking_consistency.png")

# ============================================================================
# 9. SHAP DEPENDENCE PLOTS (Top 4 Features)
# ============================================================================
print("\n" + "-"*60)
print("9. SHAP DEPENDENCE PLOTS (Top 4 Features)")
print("-"*60)

top4_features = shap_importance_df.head(4)['feature'].tolist()

for i, feature in enumerate(top4_features):
    fig_dep, ax_dep = plt.subplots(figsize=(10, 8))

    feature_idx = list(X_sample.columns).index(feature)
    shap.dependence_plot(feature_idx, shap_values_class1, X_sample,
                         ax=ax_dep, show=False)
    ax_dep.set_title(f'SHAP Dependence: {feature}', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'fig12_{i+1}_shap_dependence_{feature}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: fig12_{i+1}_shap_dependence_{feature}.png")

# ============================================================================
# 10. BUSINESS INTERPRETATION SUMMARY
# ============================================================================
print("\n" + "="*80)
print("BUSINESS INTERPRETATION SUMMARY")
print("="*80)

print("\n📊 TOP 10 MOST INFLUENTIAL FEATURES:")
print("-"*60)

for rank, (idx, row) in enumerate(shap_importance_df.head(10).iterrows(), 1):
    feature = row['feature']
    shap_val = row['mean_shap']

    # Get gain importance rank
    gain_rank_val = gain_rank[gain_rank['feature'] == feature]['gain_rank'].values[0]

    print(f"\n{rank}. {feature}")
    print(f"   SHAP Importance: {shap_val:.4f}")
    print(f"   Gain Rank: #{gain_rank_val}")

print("\n" + "-"*60)
print("CONSISTENCY CHECK:")
print("-"*60)
print(f"Spearman Correlation (Gain vs SHAP): {corr:.3f}")
if corr > 0.7:
    print("✅ High consistency - Model decision logic is stable")
elif corr > 0.5:
    print("⚠️ Moderate consistency - Some variation in feature influence")
else:
    print("❌ Low consistency - Feature importance varies by method")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE & SHAP ANALYSIS COMPLETE!")
print("="*80)

print("\nSaved Files:")
print("  CSV Files:")
print("    - feature_importance_bigdata.csv")
print("    - shap_importance_bigdata.csv")
print("\n  Visualization Files:")
print("    - fig6_feature_importance.png")
print("    - fig7_shap_importance_bar.png")
print("    - fig8_shap_summary_beeswarm.png")
print("    - fig9_shap_bar_builtin.png")
print("    - fig10_gain_vs_shap_comparison.png")
print("    - fig11_ranking_consistency.png")
for i, feature in enumerate(top4_features):
    print(f"    - fig12_{i+1}_shap_dependence_{feature}.png")

### Grey Visualisation

In [ ]:
# ============================================================================
# VISUALIZATION - PROFESSIONAL STYLE FOR TECHNICAL REPORT
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS (Professional Style)")
print("="*80)

from sklearn.metrics import precision_recall_curve, roc_curve
import matplotlib.pyplot as plt

# Professional color scheme
MAIN_COLOR = '#2C3E50'      # Dark blue-gray
SECONDARY_COLOR = '#7F8C8D' # Gray
ACCENT_COLOR = '#1A5276'    # Navy blue
LIGHT_COLOR = '#AEB6BF'     # Light gray
BASELINE_COLOR = '#E74C3C'  # Red (for baseline only)

# Set global style
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

results_big = {
    'PR_AUC': average_precision_score(y_test_big, y_proba_big),
    'F2_Score': fbeta_score(y_test_big, y_pred_big, beta=2),
    'MCC': matthews_corrcoef(y_test_big, y_pred_big),
    'Recall': recall_score(y_test_big, y_pred_big),
    'Precision': precision_score(y_test_big, y_pred_big),
    'Accuracy': accuracy_score(y_test_big, y_pred_big),
    'ROC_AUC': roc_auc_score(y_test_big, y_proba_big)
}

# ============================================================================
# 1. PR-AUC Curve (WITH Baseline)
# ============================================================================
fig1, ax1 = plt.subplots(figsize=(8, 6))

precision_vals, recall_vals, _ = precision_recall_curve(y_test_big, y_proba_big)

ax1.plot(recall_vals, precision_vals, color=MAIN_COLOR, linewidth=2,
         label=f'LGBM (PR-AUC = {results_big["PR_AUC"]:.4f})')
ax1.fill_between(recall_vals, precision_vals, alpha=0.15, color=MAIN_COLOR)

baseline = y_test_big.mean()
ax1.axhline(y=baseline, color=BASELINE_COLOR, linestyle='--', linewidth=1.5,
            label=f'Baseline = {baseline:.4f}')

ax1.set_xlabel('Recall', fontsize=11)
ax1.set_ylabel('Precision', fontsize=11)
ax1.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax1.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig1_pr_curve_with_baseline.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig1_pr_curve_with_baseline.png")

# ============================================================================
# 1b. PR-AUC Curve (WITHOUT Baseline)
# ============================================================================
fig1b, ax1b = plt.subplots(figsize=(8, 6))

ax1b.plot(recall_vals, precision_vals, color=MAIN_COLOR, linewidth=2,
         label=f'LGBM (PR-AUC = {results_big["PR_AUC"]:.4f})')
ax1b.fill_between(recall_vals, precision_vals, alpha=0.15, color=MAIN_COLOR)

ax1b.set_xlabel('Recall', fontsize=11)
ax1b.set_ylabel('Precision', fontsize=11)
ax1b.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax1b.legend(loc='upper right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax1b.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax1b.set_xlim([0, 1])
ax1b.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig1_pr_curve_no_baseline.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig1_pr_curve_no_baseline.png")

# ============================================================================
# 2. ROC Curve (WITH Baseline)
# ============================================================================
fig2, ax2 = plt.subplots(figsize=(8, 6))

fpr, tpr, _ = roc_curve(y_test_big, y_proba_big)

ax2.plot(fpr, tpr, color=MAIN_COLOR, linewidth=2,
         label=f'LGBM (ROC-AUC = {results_big["ROC_AUC"]:.4f})')
ax2.fill_between(fpr, tpr, alpha=0.15, color=MAIN_COLOR)
ax2.plot([0, 1], [0, 1], color=BASELINE_COLOR, linestyle='--', linewidth=1.5,
         label='Random Classifier')

ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax2.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig2_roc_curve_with_baseline.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig2_roc_curve_with_baseline.png")

# ============================================================================
# 2b. ROC Curve (WITHOUT Baseline)
# ============================================================================
fig2b, ax2b = plt.subplots(figsize=(8, 6))

ax2b.plot(fpr, tpr, color=MAIN_COLOR, linewidth=2,
         label=f'LGBM (ROC-AUC = {results_big["ROC_AUC"]:.4f})')
ax2b.fill_between(fpr, tpr, alpha=0.15, color=MAIN_COLOR)

ax2b.set_xlabel('False Positive Rate', fontsize=11)
ax2b.set_ylabel('True Positive Rate', fontsize=11)
ax2b.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2b.legend(loc='lower right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax2b.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
ax2b.set_xlim([0, 1])
ax2b.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('fig2_roc_curve_no_baseline.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig2_roc_curve_no_baseline.png")

# ============================================================================
# 3. Key Metrics Bar Chart
# ============================================================================
fig3, ax3 = plt.subplots(figsize=(8, 6))

metrics = ['PR-AUC', 'F2-Score', 'MCC', 'Recall']
values = [results_big['PR_AUC'], results_big['F2_Score'],
          results_big['MCC'], results_big['Recall']]

bars = ax3.bar(metrics, values, color=MAIN_COLOR, edgecolor='black', linewidth=0.8, width=0.6)

for bar, val in zip(bars, values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10)

ax3.set_ylabel('Score', fontsize=11)
ax3.set_title('Model Performance Metrics', fontsize=12, fontweight='bold')
ax3.set_ylim(0, 1.0)
ax3.grid(axis='y', alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig3_key_metrics.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig3_key_metrics.png")

# ============================================================================
# 4. Small vs Big Data Comparison
# ============================================================================
fig4, ax4 = plt.subplots(figsize=(8, 6))

metrics_compare = ['PR-AUC', 'F2-Score', 'MCC', 'Recall']
small_values = [best_model['PR_AUC'], best_model['F2_Score'],
                best_model['MCC'], best_model['Recall']]
big_values = [results_big['PR_AUC'], results_big['F2_Score'],
              results_big['MCC'], results_big['Recall']]

x = np.arange(len(metrics_compare))
width = 0.35

bars1 = ax4.bar(x - width/2, small_values, width, label='Small Data',
                color=LIGHT_COLOR, edgecolor='black', linewidth=0.8)
bars2 = ax4.bar(x + width/2, big_values, width, label='Big Data',
                color=MAIN_COLOR, edgecolor='black', linewidth=0.8)

for bar in bars1:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax4.set_ylabel('Score', fontsize=11)
ax4.set_title('Performance Comparison: Small vs Big Data', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics_compare)
ax4.legend(loc='upper right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax4.set_ylim(0, 1.0)
ax4.grid(axis='y', alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig4_small_vs_big_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig4_small_vs_big_comparison.png")

# ============================================================================
# 5. Confusion Matrix Heatmap
# ============================================================================
fig5, ax5 = plt.subplots(figsize=(6, 5))

im = ax5.imshow(cm_big, cmap='Greys')

for i in range(2):
    for j in range(2):
        color = 'white' if cm_big[i, j] > cm_big.max()/2 else 'black'
        ax5.text(j, i, f'{cm_big[i, j]:,}', ha='center', va='center',
                 fontsize=16, fontweight='bold', color=color)

ax5.set_xticks([0, 1])
ax5.set_yticks([0, 1])
ax5.set_xticklabels(['Predicted 0', 'Predicted 1'], fontsize=10)
ax5.set_yticklabels(['Actual 0', 'Actual 1'], fontsize=10)
ax5.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.colorbar(im, ax=ax5, shrink=0.8)
plt.tight_layout()
plt.savefig('fig5_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig5_confusion_matrix.png")

# ============================================================================
# FEATURE IMPORTANCE & SHAP ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE & SHAP ANALYSIS")
print("="*80)

import shap

# Extract feature importances
feature_importances_list = []
for pipeline in model_big.estimators_:
    classifier = pipeline.steps[-1][1]
    feature_importances_list.append(classifier.feature_importances_)

avg_importance = np.mean(feature_importances_list, axis=0)

feature_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': avg_importance
}).sort_values('importance', ascending=False)

print(f"\nTop 20 Features by Gain Score:")
print(feature_importance_df.head(20).to_string(index=False))

feature_importance_df.to_csv('feature_importance_bigdata.csv', index=False)
print("\n✓ Saved: feature_importance_bigdata.csv")

# SHAP Analysis
lgbm_single = model_big.estimators_[0].steps[-1][1]
explainer = shap.TreeExplainer(lgbm_single)

sample_size = min(1000, len(X_test_big_selected))
np.random.seed(random_seed)
sample_idx = np.random.choice(len(X_test_big_selected), sample_size, replace=False)
X_sample = X_test_big_selected.iloc[sample_idx]

print(f"Calculating SHAP values for {sample_size} samples...")
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

mean_shap = np.abs(shap_values_class1).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'feature': selected_features,
    'mean_shap': mean_shap
}).sort_values('mean_shap', ascending=False)

print(f"\nTop 20 Features by Mean |SHAP|:")
print(shap_importance_df.head(20).to_string(index=False))

shap_importance_df.to_csv('shap_importance_bigdata.csv', index=False)
print("\n✓ Saved: shap_importance_bigdata.csv")

# ============================================================================
# 6. Feature Importance Bar Plot (Top 20)
# ============================================================================
fig6, ax6 = plt.subplots(figsize=(8, 8))

top20_importance = feature_importance_df.head(20)

ax6.barh(range(len(top20_importance)), top20_importance['importance'].values,
         color=MAIN_COLOR, edgecolor='black', linewidth=0.5)
ax6.set_yticks(range(len(top20_importance)))
ax6.set_yticklabels(top20_importance['feature'].values, fontsize=9)
ax6.invert_yaxis()
ax6.set_xlabel('Gain Score', fontsize=11)
ax6.set_title('Feature Importance (Top 20)', fontsize=12, fontweight='bold')
ax6.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig6_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig6_feature_importance.png")

# ============================================================================
# 7. SHAP Importance Bar Plot (Top 20)
# ============================================================================
fig7, ax7 = plt.subplots(figsize=(8, 8))

top20_shap = shap_importance_df.head(20)

ax7.barh(range(len(top20_shap)), top20_shap['mean_shap'].values,
         color=MAIN_COLOR, edgecolor='black', linewidth=0.5)
ax7.set_yticks(range(len(top20_shap)))
ax7.set_yticklabels(top20_shap['feature'].values, fontsize=9)
ax7.invert_yaxis()
ax7.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax7.set_title('SHAP Feature Importance (Top 20)', fontsize=12, fontweight='bold')
ax7.grid(axis='x', alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig7_shap_importance_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig7_shap_importance_bar.png")

# ============================================================================
# 8. SHAP Summary Plot (Beeswarm)
# ============================================================================
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_class1, X_sample, plot_type="dot",
                  max_display=20, show=False, color_bar=True)
plt.title('SHAP Summary Plot', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_shap_summary_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig8_shap_summary_beeswarm.png")

# ============================================================================
# 9. SHAP Bar Plot (Built-in)
# ============================================================================
plt.figure(figsize=(8, 8))
shap.summary_plot(shap_values_class1, X_sample, plot_type="bar",
                  max_display=20, show=False, color=MAIN_COLOR)
plt.title('SHAP Feature Importance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_shap_bar_builtin.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig9_shap_bar_builtin.png")

# ============================================================================
# 10. Gain vs SHAP Comparison (Top 15)
# ============================================================================
fig10, ax10 = plt.subplots(figsize=(10, 6))

top15_features = feature_importance_df.head(15)['feature'].tolist()

shap_vals_top15 = []
for f in top15_features:
    val = shap_importance_df[shap_importance_df['feature'] == f]['mean_shap'].values
    shap_vals_top15.append(val[0] if len(val) > 0 else 0)

importance_vals_top15 = feature_importance_df.head(15)['importance'].values

importance_norm = importance_vals_top15 / importance_vals_top15.max()
shap_norm = np.array(shap_vals_top15) / max(shap_vals_top15)

x = np.arange(len(top15_features))
width = 0.35

bars1 = ax10.bar(x - width/2, importance_norm, width, label='Gain (normalized)',
                 color=LIGHT_COLOR, edgecolor='black', linewidth=0.5)
bars2 = ax10.bar(x + width/2, shap_norm, width, label='SHAP (normalized)',
                 color=MAIN_COLOR, edgecolor='black', linewidth=0.5)

ax10.set_xticks(x)
ax10.set_xticklabels(top15_features, rotation=45, ha='right', fontsize=8)
ax10.set_ylabel('Normalized Importance', fontsize=11)
ax10.set_title('Gain vs SHAP Importance Comparison', fontsize=12, fontweight='bold')
ax10.legend(loc='upper right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax10.grid(axis='y', alpha=0.3, linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig10_gain_vs_shap_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig10_gain_vs_shap_comparison.png")

# ============================================================================
# 11. Ranking Consistency
# ============================================================================
fig11, ax11 = plt.subplots(figsize=(8, 6))

gain_rank = feature_importance_df.reset_index(drop=True).copy()
gain_rank['gain_rank'] = range(1, len(gain_rank) + 1)

shap_rank = shap_importance_df.reset_index(drop=True).copy()
shap_rank['shap_rank'] = range(1, len(shap_rank) + 1)

rank_comparison = gain_rank.merge(shap_rank[['feature', 'shap_rank']], on='feature')
rank_comparison = rank_comparison.head(30)

ax11.scatter(rank_comparison['gain_rank'], rank_comparison['shap_rank'],
             s=60, c=MAIN_COLOR, edgecolor='black', alpha=0.7)

max_rank = max(rank_comparison['gain_rank'].max(), rank_comparison['shap_rank'].max())
ax11.plot([1, max_rank], [1, max_rank], color=BASELINE_COLOR, linestyle='--',
          linewidth=1.5, label='Perfect Agreement')

for idx, row in rank_comparison.head(10).iterrows():
    ax11.annotate(row['feature'], (row['gain_rank'], row['shap_rank']),
                  fontsize=7, ha='left', va='bottom')

ax11.set_xlabel('Gain Importance Rank', fontsize=11)
ax11.set_ylabel('SHAP Importance Rank', fontsize=11)
ax11.set_title('Ranking Consistency: Gain vs SHAP', fontsize=12, fontweight='bold')
ax11.legend(loc='lower right', fontsize=9, frameon=True, fancybox=False, edgecolor='black')
ax11.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)

from scipy.stats import spearmanr
corr, pval = spearmanr(rank_comparison['gain_rank'], rank_comparison['shap_rank'])
ax11.text(0.05, 0.95, f'Spearman ρ = {corr:.3f}\np = {pval:.4f}',
          transform=ax11.transAxes, fontsize=10, verticalalignment='top',
          bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', alpha=0.8))

plt.tight_layout()
plt.savefig('fig11_ranking_consistency.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: fig11_ranking_consistency.png")

# ============================================================================
# 12. SHAP Dependence Plots (Top 4 Features)
# ============================================================================
print("\n" + "-"*60)
print("SHAP Dependence Plots (Top 4 Features)")
print("-"*60)

top4_features = shap_importance_df.head(4)['feature'].tolist()

for i, feature in enumerate(top4_features):
    fig_dep, ax_dep = plt.subplots(figsize=(8, 6))

    feature_idx = list(X_sample.columns).index(feature)
    shap.dependence_plot(feature_idx, shap_values_class1, X_sample,
                         ax=ax_dep, show=False)
    ax_dep.set_title(f'SHAP Dependence: {feature}', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'fig12_{i+1}_shap_dependence_{feature}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: fig12_{i+1}_shap_dependence_{feature}.png")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("ALL VISUALIZATIONS COMPLETE!")
print("="*80)

print("\nSaved Files:")
print("  - fig1_pr_curve_with_baseline.png")
print("  - fig1_pr_curve_no_baseline.png")
print("  - fig2_roc_curve_with_baseline.png")
print("  - fig2_roc_curve_no_baseline.png")
print("  - fig3_key_metrics.png")
print("  - fig4_small_vs_big_comparison.png")
print("  - fig5_confusion_matrix.png")
print("  - fig6_feature_importance.png")
print("  - fig7_shap_importance_bar.png")
print("  - fig8_shap_summary_beeswarm.png")
print("  - fig9_shap_bar_builtin.png")
print("  - fig10_gain_vs_shap_comparison.png")
print("  - fig11_ranking_consistency.png")
for i, feature in enumerate(top4_features):
    print(f"  - fig12_{i+1}_shap_dependence_{feature}.png")

print("\n" + "="*80)